# 5. Optimización simbólica de las geodésicas de Kerr: métrica corregida

## 5.0. Introducción y alcance

Este cuaderno deriva la conexión desde la **métrica corregida de producción**, aplica eliminación de subexpresiones comunes (CSE) después de contraerla y emite dos funciones C99 experimentales. Reproduce la estructura algebraica del cuaderno 004, reescribiendo su lógica sin cargar el snapshot legado. Firma $-+++$, coordenadas Boyer–Lindquist $(t,r,\theta,\phi)$, $G=c=1$, $M=\mathtt{mass}$ y $a=\mathtt{spin}$ (longitud de spin, no $a/M$). La API normalizada de la arquitectura usa $M=1$; aquí se conserva $M$ explícito para verificar la geometría.

> **Candidato no adoptado.** Solo documentación y experimento. La decisión 7 de arquitectura §14 (expresiones analíticas, código generado y verificación de Christoffel) sigue **PENDIENTE**. No se modifica `relatipy/`. La selección de raíz dentro de la ergosfera es una propuesta experimental; el marco local de observador sigue pendiente. Las secciones 5.8 y 5.9 comparan y miden el candidato emitido aquí.

La evidencia es local y ejecutable, pendiente de revisión humana. Identificadores usados en este cuaderno: **S-C** = `relatipy/native/src/metric/physic/kerr.c`, funciones y líneas registradas en 5.2; **S-A** = `arquitectura/arquitectura.md` §§6.3, 6.13 y 14; **S-E** = `SCRATCH/common_contract.md`, firmas y entorno. **C-METRIC** (reportado, S-C: `rp_kerr_physic_prepare_point` y `rp_kerr_physic_evaluate_metric`) identifica las expresiones transcritas. **C-GAMMA** (derivado de C-METRIC, 5.3–5.7) identifica la conexión y su contracción. **C-NULL** (derivado de C-METRIC y S-A §6.13, 5.10) identifica la cuadrática y el análisis de raíces. Los valores de las tablas son cálculos propios, no mediciones bibliográficas ni afirmaciones verificadas manualmente.

**Ejecución de las salidas guardadas.** En esta fase, el comando `uv` estándar del contrato dio **FAIL ambiental** por caché de solo lectura; usando caché temporal y paquetes locales, el inicio del kernel estándar también falló porque el sandbox prohíbe enlazar sockets (`PermissionError: [Errno 1] Operation not permitted`). Se conserva el mismo entorno de versiones, `nbclient`, `timeout=1800` y `kernel_name="python3"`; el adaptador temporal `SCRATCH/execute_notebook.py` usa `InProcessKernelManager`, canales en memoria y publicación síncrona. `SCRATCH/run_notebook.sh` ejecuta `uv run --offline --no-project --python 3.14` con los mismos `--with` del contrato; caché, IPython y temporales viven en SCRATCH. La alternativa permite validar el contenido y guardar las salidas, pero no convierte el fallo del comando estándar en PASS. El cuaderno no depende del adaptador para sus cálculos ni requiere modificar `.venv`.

**Estado del árbol compartido.** Otros trabajadores modifican `relatipy/` en paralelo. La comprobación final exige SHA-256 idénticos para las seis fuentes C de producción leídas y compiladas; un cambio en ellas detiene la ejecución. `git status --short` se muestra solo como información y sus cambios ajenos no provocan una excepción. Este frente no escribe en producción.


## 5.1. Sistema de primer orden

Para una tangente afín $k^\mu=dx^\mu/d\lambda$,

$$\frac{dx^\mu}{d\lambda}=k^\mu,\qquad
\frac{dk^\mu}{d\lambda}=-\Gamma^\mu{}_{\alpha\beta}k^\alpha k^\beta.$$

El RHS es **norma-agnóstico**: la misma conexión sirve para tangentes temporales y nulas. La condición $g(k,k)=-1$ o $0$ se fija en los datos iniciales, no en esta contracción. El nombre contractual `rp_candidate_kerr_null_acceleration` no restringe su identidad algebraica a fotones. El estado sigue el orden de ocho componentes de §§6.3/6.13; $k^t=1$ solo se impone inicialmente en la rama nula, no durante la evolución.


In [1]:
from __future__ import annotations

import ctypes
import hashlib
import json
import platform
import os
import shutil
import subprocess
import tempfile
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'relatipy/native/src/metric/physic/kerr.c').is_file():
            return candidate
    raise FileNotFoundError('No se encuentra la raíz del proyecto')

ROOT = find_project_root()
# Exportación optativa; la ejecución ordinaria escribe solo en temporales.
export_directory_value = os.environ.get('RELATIPY_005_EXPORT_DIR')
EXPORT_DIR = Path(export_directory_value).expanduser().resolve() if export_directory_value else None
if EXPORT_DIR is not None:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
production_sources = [ROOT/'relatipy/native/src'/name for name in (
    'geodesic/kerr.c', 'geodesic/physic/kerr.c', 'metric/kerr.c',
    'metric/physic/kerr.c', 'utils/numeric.c', 'utils/tensor.c',
)]
def production_hashes():
    """Identifica las seis unidades C leídas y compiladas en este experimento."""
    return {str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
            for path in production_sources}
production_hashes_before = production_hashes()
compiler = shutil.which('cc')
if compiler is None:
    raise RuntimeError('Se requiere cc; esta validación no permite SKIP')
pointer = ctypes.POINTER(ctypes.c_double)
status_before = subprocess.check_output(['git', '-C', str(ROOT/'relatipy'), 'status', '--short'], text=True)
SEED = 20260930
rng = np.random.default_rng(SEED)
pd.DataFrame([{
    'Python': platform.python_version(), 'SymPy': sp.__version__,
    'NumPy': np.__version__, 'pandas': pd.__version__, 'semilla': SEED,
    'cc': subprocess.check_output([compiler, '--version'], text=True).splitlines()[0],
    'producción': 'solo lectura', 'revisión humana': 'pending',
}])


,Python,SymPy,NumPy,pandas,semilla,cc,producción,revisión humana
0,3.14.7,1.14.0,2.5.2,3.0.6,20260930,cc (GCC) 16.2.1 20260810,solo lectura,pending


## 5.2. Métrica corregida y verificación contra C

Se define independientemente

$$\Sigma=r^2+a^2\cos^2\theta,\quad\Delta=r^2-2Mr+a^2,$$
$$g_{tt}=-1+\frac{2Mr}{\Sigma},\quad
g_{t\phi}=-\frac{2Mar\sin^2\theta}{\Sigma},\quad
g_{rr}=\frac{\Sigma}{\Delta},\quad g_{\theta\theta}=\Sigma,$$
$$g_{\phi\phi}=\left(r^2+a^2+\frac{2Ma^2r\sin^2\theta}{\Sigma}\right)\sin^2\theta.$$

Después se transcriben literalmente las asignaciones de `rp_kerr_physic_prepare_point` y `rp_kerr_physic_evaluate_metric` (C-METRIC). Los literales C `1.0` y `2.0` se representan como enteros exactos de SymPy. La tabla conserva los localizadores y el SHA-256 del archivo leído; el chequeo simbólico muestra las **16 diferencias**, incluidos ceros y copia simétrica. $M,a$ son símbolos positivos reales; las sustituciones numéricas incluyen el límite $a=0$ y, como control algebraico, spin negativo. Se excluyen $\Sigma=0$, $\Delta=0$ y el eje BL.


In [2]:
metric_source = ROOT / 'relatipy/native/src/metric/physic/kerr.c'
source_text = metric_source.read_text(encoding='utf-8')
source_lines = source_text.splitlines()
source_hash = hashlib.sha256(metric_source.read_bytes()).hexdigest()
def source_location(token):
    line = next(i for i, text in enumerate(source_lines, 1) if token in text)
    return f'{metric_source.relative_to(ROOT)}:{line}'

M, a = sp.symbols('mass spin', positive=True, real=True)
t, r, th, ph = sp.symbols('t r theta phi', real=True)
k = sp.symbols('k0:4', real=True)
coordinates = (t, r, th, ph)
sigma = r**2 + a**2 * sp.cos(th)**2
delta = r**2 - 2*M*r + a**2
sin2 = sp.sin(th)**2
g_sym = sp.zeros(4)
g_sym[0, 0] = -1 + 2*M*r/sigma
g_sym[0, 3] = g_sym[3, 0] = -2*M*a*r*sin2/sigma
g_sym[1, 1] = sigma/delta
g_sym[2, 2] = sigma
g_sym[3, 3] = (r**2 + a**2 + 2*M*a**2*r*sin2/sigma)*sin2

# Transcripción S-C: rp_kerr_physic_prepare_point (sigma/delta).
radius = r
radius_squared = radius * radius
spin_squared = a * a
sin_theta = sp.sin(th)
cos_theta = sp.cos(th)
point_sigma = radius_squared + spin_squared * cos_theta * cos_theta
point_delta = radius_squared - 2 * M * radius + spin_squared

# Transcripción S-C: rp_kerr_physic_evaluate_metric, sin cambiar su orden algebraico.
sin_squared = sin_theta * sin_theta
common = 2 * M * radius / point_sigma
g_C = sp.zeros(4)  # rp_tensor_zero
g_C[0, 0] = -(1 - common)
g_C[0, 3] = -common * a * sin_squared
g_C[3, 0] = g_C[0, 3]
g_C[1, 1] = point_sigma / point_delta
g_C[2, 2] = point_sigma
g_C[3, 3] = (radius_squared + spin_squared
              + common * spin_squared * sin_squared) * sin_squared
metric_residuals = [sp.simplify(g_sym[i, j] - g_C[i, j]) for i, j in product(range(4), repeat=2)]
if any(value != 0 for value in metric_residuals):
    raise ArithmeticError(f'Métrica simbólica distinta de la transcripción: {metric_residuals}')
metric_symbolic_table = pd.DataFrame([
    {'mu': i, 'nu': j, 'g_sym': g_sym[i, j], 'g_C': g_C[i, j],
     'simplify(g_sym-g_C)': metric_residuals[4*i+j], 'claim_id': 'C-METRIC'}
    for i, j in product(range(4), repeat=2)
])
metric_symbolic_table


,mu,nu,g_sym,g_C,simplify(g_sym-g_C),claim_id
0,0,0,2*mass*r/(r**2 + spin**2*cos(theta)**2) - 1,2*mass*r/(r**2 + spin**2*cos(theta)**2) - 1,0,C-METRIC
1,0,1,0,0,0,C-METRIC
2,0,2,0,0,0,C-METRIC
3,0,3,-2*mass*r*spin*sin(theta)**2/(r**2 + spin**2*c...,-2*mass*r*spin*sin(theta)**2/(r**2 + spin**2*c...,0,C-METRIC
4,1,0,0,0,0,C-METRIC
5,1,1,(r**2 + spin**2*cos(theta)**2)/(-2*mass*r + r*...,(r**2 + spin**2*cos(theta)**2)/(-2*mass*r + r*...,0,C-METRIC
6,1,2,0,0,0,C-METRIC
7,1,3,0,0,0,C-METRIC
8,2,0,0,0,0,C-METRIC
9,2,1,0,0,0,C-METRIC


In [3]:
pd.DataFrame([
    {'source_id': 'S-C', 'localizador': source_location(token), 'SHA256': source_hash}
    for token in ('rp_kerr_physic_prepare_point(', 'point->sigma = radius_squared',
                  'point->delta = radius_squared', 'rp_kerr_physic_evaluate_metric(',
                  'metric[0][0] = -(1.0 - common);', 'metric[3][3] = (radius_squared')
])


,source_id,localizador,SHA256
0,S-C,relatipy/native/src/metric/physic/kerr.c:34,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...
1,S-C,relatipy/native/src/metric/physic/kerr.c:55,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...
2,S-C,relatipy/native/src/metric/physic/kerr.c:57,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...
3,S-C,relatipy/native/src/metric/physic/kerr.c:79,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...
4,S-C,relatipy/native/src/metric/physic/kerr.c:93,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...
5,S-C,relatipy/native/src/metric/physic/kerr.c:98,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...


In [4]:
# Producción compilada fuera del repositorio; el objeto temporal vive durante el kernel.
production_temp = tempfile.TemporaryDirectory(prefix='relatipy-corrected-metric-')
production_library_path = Path(production_temp.name) / 'libmetric.so'
production_command = [compiler, '-std=c11', '-O3', '-fPIC', '-shared',
    '-I', str(ROOT/'relatipy/native/include'), '-I', str(ROOT/'relatipy/native/src'),
    *map(str, production_sources), '-lm', '-o', str(production_library_path)]
subprocess.run(production_command, check=True, capture_output=True, text=True)
production_library = ctypes.CDLL(str(production_library_path))
production_metric = production_library.rp_kerr_metric
production_metric.argtypes = [ctypes.c_double, ctypes.c_double, pointer, pointer]
production_metric.restype = ctypes.c_int

def native_metric(mass, spin, coordinates):
    x_buffer = (ctypes.c_double * 4)(*coordinates)
    result = (ctypes.c_double * 16)()
    status = production_metric(mass, spin, x_buffer, result)
    if status != 0:
        raise RuntimeError(f'rp_kerr_metric: status={status}, M={mass}, a={spin}, x={coordinates}')
    return np.ctypeslib.as_array(result).reshape(4, 4).copy()

metric_function = sp.lambdify((M, a, r, th), g_sym, modules='numpy')
metric_samples = (
    (1.0, 0.0, 8.0, 1.1), (1.0, 0.5, 8.0, 1.1),
    (2.0, 1.8, 5.0, 0.8), (1.0, 0.99, 1.2, 1.4),
    (1.0, 1.0, 1.01, 0.2), (1.0, -0.5, 15.0, 2.1),
)
metric_native_rows = []
for mass, spin, radius, theta in metric_samples:
    expected = np.asarray(metric_function(mass, spin, radius, theta), dtype=float)
    actual = native_metric(mass, spin, (0.0, radius, theta, 0.0))
    np.testing.assert_allclose(actual, expected, rtol=3e-13, atol=3e-14)
    metric_native_rows.append({'M': mass, 'a': spin, 'r': radius, 'theta': theta,
        'max_error_absoluto': float(np.max(np.abs(actual-expected))), 'estado': 'PASS'})
metric_native_table = pd.DataFrame(metric_native_rows)
metric_native_table


,M,a,r,theta,max_error_absoluto,estado
0,1.0,0.00,8.00,1.1,0.000000e+00,PASS
1,1.0,0.50,8.00,1.1,0.000000e+00,PASS
2,2.0,1.80,5.00,0.8,1.776357e-15,PASS
3,1.0,0.99,1.20,1.4,0.000000e+00,PASS
4,1.0,1.00,1.01,0.2,6.938894e-18,PASS
5,1.0,-0.50,15.00,2.1,6.938894e-18,PASS


## 5.3. Christoffel desde esa métrica

Se usa la inversa analítica del bloque $(t,\phi)$, junto con $g^{rr}=\Delta/\Sigma$ y $g^{\theta\theta}=1/\Sigma$:

$$g^{tt}=-\frac{(r^2+a^2)^2-a^2\Delta\sin^2\theta}{\Sigma\Delta},\quad
g^{t\phi}=-\frac{2Mar}{\Sigma\Delta},\quad
g^{\phi\phi}=\frac{\Delta-a^2\sin^2\theta}{\Sigma\Delta\sin^2\theta}.$$

Se comprueba $g g^{-1}=I$ antes de derivar

$$\Gamma^\mu{}_{\alpha\beta}=\tfrac12g^{\mu\nu}
(\partial_\alpha g_{\nu\beta}+\partial_\beta g_{\nu\alpha}-\partial_\nu g_{\alpha\beta}).$$

La simplificación es racional: reduce potencias de $\sin\theta$ módulo $\sin^2\theta+\cos^2\theta=1$ y factoriza. Se reescriben $(\cos\theta-1)(\cos\theta+1)=-\sin^2\theta$ para evitar cancelación polar artificial. Se verifican también las 64 identidades respecto a la fórmula cruda. La comparación opcional con `rp_kerr_christoffel` se omite: S-C, cabecera del archivo y función `rp_kerr_physic_evaluate_christoffel`, declaran que conserva expresiones legadas; no es una referencia corregida. La comparación relevante con el RHS productivo se ejecuta en 5.8.


In [5]:
def simplify_geometry(expression):
    sine, cosine = sp.symbols('_sine _cosine')
    algebraic = expression.xreplace({sp.sin(th): sine, sp.cos(th): cosine})
    numerator, denominator = sp.fraction(sp.cancel(algebraic))
    def reduce_sine(polynomial):
        return sum(coefficient * sine**(power[0] % 2) * (1-cosine**2)**(power[0]//2)
                   for power, coefficient in sp.Poly(polynomial, sine).terms())
    result = sp.factor(sp.cancel(reduce_sine(numerator)/reduce_sine(denominator)))
    result = result.subs((cosine-1)*(cosine+1), -sine**2)
    result = result.subs((-a*cosine+r)*(a*cosine+r), r**2-a**2*cosine**2)
    return result.xreplace({sine: sp.sin(th), cosine: sp.cos(th)})

inverse_metric = sp.zeros(4)
inverse_metric[0, 0] = -((r**2+a**2)**2-a**2*delta*sin2)/(sigma*delta)
inverse_metric[0, 3] = inverse_metric[3, 0] = -2*M*a*r/(sigma*delta)
inverse_metric[1, 1] = delta/sigma
inverse_metric[2, 2] = 1/sigma
inverse_metric[3, 3] = (delta-a**2*sin2)/(sigma*delta*sin2)
inverse_residuals = [simplify_geometry(value) for value in g_sym*inverse_metric-sp.eye(4)]
if any(value != 0 for value in inverse_residuals):
    raise ArithmeticError(f'Inversa incorrecta: {inverse_residuals}')
pd.DataFrame([{'mu': i, 'nu': j, '(g*g_inverse-I)_mu_nu': inverse_residuals[4*i+j]}
              for i, j in product(range(4), repeat=2)])


,mu,nu,(g*g_inverse-I)_mu_nu
0,0,0,0
1,0,1,0
2,0,2,0
3,0,3,0
4,1,0,0
5,1,1,0
6,1,2,0
7,1,3,0
8,2,0,0
9,2,1,0


In [6]:
Gamma = sp.MutableDenseNDimArray.zeros(4, 4, 4)
gamma_raw = sp.MutableDenseNDimArray.zeros(4, 4, 4)
for mu, alpha, beta in product(range(4), repeat=3):
    raw = sp.Rational(1, 2)*sum(
        inverse_metric[mu, nu] * (
            sp.diff(g_sym[nu, beta], coordinates[alpha])
            + sp.diff(g_sym[nu, alpha], coordinates[beta])
            - sp.diff(g_sym[alpha, beta], coordinates[nu])) for nu in range(4))
    gamma_raw[mu, alpha, beta] = raw
    Gamma[mu, alpha, beta] = simplify_geometry(raw)
gamma_symmetry_residuals = [simplify_geometry(Gamma[mu, alpha, beta]-Gamma[mu, beta, alpha])
    for mu, alpha, beta in product(range(4), repeat=3)]
gamma_formula_residuals = [simplify_geometry(Gamma[mu, alpha, beta]-gamma_raw[mu, alpha, beta])
    for mu, alpha, beta in product(range(4), repeat=3)]
if any(value != 0 for value in (*gamma_symmetry_residuals, *gamma_formula_residuals)):
    raise ArithmeticError('Falló la identidad de Christoffel o su simetría')
gamma_table = pd.DataFrame([{
    'mu': mu, 'alpha': alpha, 'beta': beta, 'Gamma': Gamma[mu, alpha, beta],
    'residuo_fórmula': gamma_formula_residuals[16*mu+4*alpha+beta],
    'residuo_simetría': gamma_symmetry_residuals[16*mu+4*alpha+beta],
} for mu, alpha, beta in product(range(4), repeat=3) if alpha <= beta])
gamma_table


,mu,alpha,beta,Gamma,residuo_fórmula,residuo_simetría
0,0,0,0,0,0,0
1,0,0,1,mass*(r**2 + spin**2)*(r**2 - spin**2*cos(thet...,0,0
2,0,0,2,-2*mass*r*spin**2*sin(theta)*cos(theta)/(r**2 ...,0,0
3,0,0,3,0,0,0
4,0,1,1,0,0,0
5,0,1,2,0,0,0
6,0,1,3,-mass*spin*(3*r**4 + r**2*spin**2*cos(theta)**...,0,0
7,0,2,2,0,0,0
8,0,2,3,2*mass*r*spin**3*sin(theta)**3*cos(theta)/(r**...,0,0
9,0,3,3,0,0,0


## 5.4. Contracción simétrica antes de optimizar

Como en 004 §4.4, los términos diagonales se evalúan una vez y los pares fuera de diagonal llevan factor dos. Se compara con la contracción completa de 16 términos por componente. No se usa la condición nula para eliminar componentes ni se presupone $k^t=1$.


In [7]:
direct_acceleration = tuple(-sum(Gamma[mu, alpha, beta]*k[alpha]*k[beta]
    for alpha, beta in product(range(4), repeat=2)) for mu in range(4))
symmetric_acceleration = tuple(
    -sum(Gamma[mu, alpha, alpha]*k[alpha]**2 for alpha in range(4))
    -2*sum(Gamma[mu, alpha, beta]*k[alpha]*k[beta]
           for alpha in range(4) for beta in range(alpha+1, 4)) for mu in range(4))
contraction_residuals = tuple(sp.simplify(direct-symmetric)
    for direct, symmetric in zip(direct_acceleration, symmetric_acceleration))
if contraction_residuals != (0, 0, 0, 0):
    raise ArithmeticError(f'Contracción incorrecta: {contraction_residuals}')
first_order_rhs = (*k, *symmetric_acceleration)
pd.DataFrame([{'mu': mu, 'residuo_contracción': contraction_residuals[mu],
               'operaciones_sin_CSE': int(sp.count_ops(expression))}
              for mu, expression in enumerate(symmetric_acceleration)])


,mu,residuo_contracción,operaciones_sin_CSE
0,0,0,110
1,1,0,182
2,2,0,164
3,3,0,150


## 5.5. Estrategias CSE, selección reproducible y nombres semánticos

Se reescriben las cuatro estrategias de 004 §4.5: CSE directa y después de `factor_terms`, ambas sin optimizaciones o con `basic`. El costo suma `count_ops` en auxiliares y salidas. Desempate: menos auxiliares, después nombre lexicográfico; el tiempo medido **no** interviene. Orden `canonical`, versiones registradas y semilla fija permiten reproducir el resultado; este costo algebraico no es un benchmark ni demuestra optimalidad global.

El ganador se renombra con significados algebraicos (`sigma`, `delta`, `r2`, `spin2`, `cos_th`, etc.). La lista explícita corresponde a las definiciones mostradas, no a posiciones opacas: se valida su expansión en 5.6. Si una versión de SymPy cambia el ganador, la guardia exige revisar el mapa en lugar de emitir silenciosamente nombres erróneos.


In [8]:
def operation_cost(replacements, expressions):
    return int(sum(sp.count_ops(value) for _, value in replacements)
               + sum(sp.count_ops(expression) for expression in expressions))

candidate_inputs = {
    'cse': symmetric_acceleration,
    'factor_terms+cse': tuple(sp.factor_terms(expression) for expression in symmetric_acceleration),
}
strategies = []
for input_name, expressions in candidate_inputs.items():
    for optimization_name, optimizations in (('raw', None), ('basic', 'basic')):
        started = time.perf_counter()
        replacements, reduced = sp.cse(expressions, symbols=sp.numbered_symbols('cse_aux'),
                                       optimizations=optimizations, order='canonical')
        strategies.append({'name': f'{input_name}:{optimization_name}',
            'replacements': tuple(replacements), 'reduced': tuple(reduced),
            'temporaries': len(replacements), 'operations': operation_cost(replacements, reduced),
            'elapsed_s': time.perf_counter()-started})
best_cse = min(strategies, key=lambda item: (item['operations'], item['temporaries'], item['name']))
baseline_operations = int(sum(sp.count_ops(expression) for expression in symmetric_acceleration))
semantic_names = (
    'two_k2', 'spin2', 'spin2_r', 'cos_th', 'sin_th', 'cos_th_sin_th',
    'spin2_r_cos_th_sin_th', 'cos_th_k3', 'r2', 'r2_plus_spin2', 'two_mass',
    'two_mass_r', 'delta', 'inv_delta', 'k1_over_delta', 'cos2_th',
    'spin2_cos2_th', 'r2_minus_spin2_cos2_th', 'r2_minus_spin2_cos2_th_k0',
    'r2_minus_spin2_cos2_th_k0_k1_over_delta', 'r4', 'spin2_r2', 'spin4',
    'spin4_cos2_th', 'spin2_cos2_th_r2', 'k1_k3_over_delta', 'sin2_th',
    'spin_sin2_th', 'sigma', 'inv_sigma2', 'two_mass_over_sigma2', 'inv_sigma',
    'two_k1_k2', 'spin2_cos_th_sin_th', 'k2_squared', 'k0_squared',
    'delta_over_sigma2', 'mass_r2', 'mass_spin2_cos2_th', 'k1_squared_over_delta',
    'k3_squared', 'r5', 'mass_spin2_r2', 'mass_spin4_cos2_th', 'cos4_th',
    'spin4_cos4_th', 'spin4_cos4_th_r', 'mass_spin4_cos4_th', 'r3',
    'two_spin2_cos2_th', 'two_spin2_cos2_th_r3', 'mass_spin2_cos2_th_r2',
    'cos_th_sin_th_over_sigma2', 'mass_spin', 'four_r3', 'k2_over_sin_th',
)
if best_cse['name'] != 'factor_terms+cse:basic' or len(best_cse['replacements']) != len(semantic_names):
    raise RuntimeError('Cambió la estructura CSE; revisar explícitamente los nombres semánticos')
semantic_symbols = tuple(sp.Symbol(name) for name in semantic_names)
semantic_substitutions = {old: new for (old, _), new in zip(best_cse['replacements'], semantic_symbols)}
semantic_replacements = tuple((semantic_substitutions[old], expression.xreplace(semantic_substitutions))
                              for old, expression in best_cse['replacements'])
semantic_acceleration = tuple(expression.xreplace(semantic_substitutions) for expression in best_cse['reduced'])
cse_strategy_table = pd.DataFrame([{
    'estrategia': item['name'], 'operaciones': item['operations'], 'auxiliares': item['temporaries'],
    'tiempo_s': item['elapsed_s'], 'seleccionada': item['name'] == best_cse['name'],
    'operaciones_sin_CSE': baseline_operations,
} for item in strategies])
cse_strategy_table


,estrategia,operaciones,auxiliares,tiempo_s,seleccionada,operaciones_sin_CSE
0,cse:raw,203,52,0.013021,False,606
1,cse:basic,188,52,0.030409,False,606
2,factor_terms+cse:raw,197,49,0.007955,False,606
3,factor_terms+cse:basic,178,56,0.024701,True,606


In [9]:
pd.DataFrame([{'nombre_semántico': str(symbol), 'definición': definition}
              for symbol, definition in semantic_replacements])


,nombre_semántico,definición
0,two_k2,2*k2
1,spin2,spin**2
2,spin2_r,r*spin2
3,cos_th,cos(theta)
4,sin_th,sin(theta)
5,cos_th_sin_th,cos_th*sin_th
6,spin2_r_cos_th_sin_th,cos_th_sin_th*spin2_r
7,cos_th_k3,cos_th*k3
8,r2,r**2
9,r2_plus_spin2,r2 + spin2


## 5.6. Equivalencia simbólica y numérica de la expansión CSE

Se sustituyen los auxiliares en orden inverso. Las cuatro diferencias deben ser exactamente cero. Después se comparan valores con `lambdify` en casos regulares y 24 casos pseudoaleatorios con orden y semilla fijos. La tangente es arbitraria: este control prueba la identidad del RHS sin imponer una norma.


In [10]:
def expand_cse(replacements, reduced):
    expanded = list(reduced)
    for symbol, definition in reversed(replacements):
        expanded = [expression.xreplace({symbol: definition}) for expression in expanded]
    return tuple(expanded)

expanded_semantic_cse = expand_cse(semantic_replacements, semantic_acceleration)
symbolic_residuals = tuple(sp.simplify(original-optimized)
    for original, optimized in zip(symmetric_acceleration, expanded_semantic_cse))
if symbolic_residuals != (0, 0, 0, 0):
    raise ArithmeticError(f'CSE no equivalente: {symbolic_residuals}')
parameters = (M, a, r, th, *k)
direct_function = sp.lambdify(parameters, symmetric_acceleration, modules='numpy')
optimized_function = sp.lambdify(parameters, expanded_semantic_cse, modules='numpy')
samples = [
    (1.0, 0.5, 8.0, 1.1, 1.2, -0.03, 0.01, 0.08),
    (2.0, -1.4, 15.0, 0.8, 1.1, 0.02, -0.015, 0.04),
    (0.75, 0.0, 6.0, 1.35, 1.3, -0.01, 0.02, -0.05),
    (1.0, 1.0, 1.05, 0.2, 1.0, -0.003, 0.02, 0.1),
]
for _ in range(24):
    mass = float(rng.uniform(0.5, 2.0))
    spin = float(rng.uniform(0.0, 1.0)*mass)
    horizon = mass + np.sqrt(mass**2-spin**2)
    samples.append((mass, spin, float(horizon+rng.uniform(0.1, 30.0)*mass),
                    float(rng.uniform(0.05, np.pi-0.05)), *map(float, rng.normal(size=4))))
cse_numeric_rows = []
for index, sample in enumerate(samples):
    expected = np.asarray(direct_function(*sample), dtype=float)
    actual = np.asarray(optimized_function(*sample), dtype=float)
    np.testing.assert_allclose(actual, expected, rtol=3e-12, atol=3e-13)
    cse_numeric_rows.append({'muestra': index, 'max_error_absoluto': float(np.max(np.abs(actual-expected))),
                             'estado': 'PASS'})
pd.DataFrame([{'mu': mu, 'residuo_simbólico': residual,
               'muestras_numéricas': len(samples),
               'max_error_numérico': max(row['max_error_absoluto'] for row in cse_numeric_rows)}
              for mu, residual in enumerate(symbolic_residuals)])


,mu,residuo_simbólico,muestras_numéricas,max_error_numérico
0,0,0,28,7.105427e-15
1,1,0,28,7.105427e-15
2,2,0,28,7.105427e-15
3,3,0,28,7.105427e-15


## 5.7. Emisión C99 y prueba nativa mínima

La firma es exactamente S-E; la función calcula únicamente $dk^\mu/d\lambda$. Se emite el texto completo con `<math.h>` en `c_acceleration` y se escribe el archivo de compilación en `tempfile.TemporaryDirectory`. Solo si se define `RELATIPY_005_EXPORT_DIR`, se exporta además `candidate_acceleration.c` a ese directorio, sin estado global, validación ni asignación dinámica. El llamador aporta buffers válidos y no solapados y un punto regular. Se valida con `cc -std=c99 -pedantic -Wall -Wextra -Werror -fsyntax-only`; se compila además una biblioteca temporal con esos controles y `-O3`, sin `fast-math`. La prueba `ctypes` compara los 28 casos anteriores con la contracción directa SymPy. No es la comparación contra producción de 5.8.


In [11]:
def emit_c_acceleration(replacements, expressions):
    lines = [
        '#include <math.h>',
        '/* Experimental candidate; not adopted in production. */',
        'void rp_candidate_kerr_null_acceleration(',
        '    double mass, double spin,',
        '    const double coordinates[4],',
        '    const double tangent[4],',
        '    double tangent_derivative[4])',
        '{',
        '    const double r = coordinates[1];',
        '    const double theta = coordinates[2];',
    ]
    lines.extend(f'    const double k{index} = tangent[{index}];' for index in range(4))
    lines.extend(f'    const double {symbol} = {sp.ccode(expression, standard="C99")};'
                 for symbol, expression in replacements)
    lines.extend(f'    tangent_derivative[{mu}] = {sp.ccode(expression, standard="C99")};'
                 for mu, expression in enumerate(expressions))
    lines.append('}')
    return '\n'.join(lines)+'\n'

c_acceleration = emit_c_acceleration(semantic_replacements, semantic_acceleration)
acceleration_temp = tempfile.TemporaryDirectory(prefix='relatipy-emitted-acceleration-')
acceleration_source = Path(acceleration_temp.name)/'candidate_acceleration.c'
acceleration_source.write_text(c_acceleration, encoding='utf-8')
if EXPORT_DIR is not None:
    (EXPORT_DIR/'candidate_acceleration.c').write_text(c_acceleration, encoding='utf-8')
strict_acceleration_command = [compiler, '-std=c99', '-pedantic', '-Wall', '-Wextra', '-Werror',
                               '-fsyntax-only', str(acceleration_source)]
subprocess.run(strict_acceleration_command, check=True, capture_output=True, text=True)
print(c_acceleration)


#include <math.h>
/* Experimental candidate; not adopted in production. */
void rp_candidate_kerr_null_acceleration(
    double mass, double spin,
    const double coordinates[4],
    const double tangent[4],
    double tangent_derivative[4])
{
    const double r = coordinates[1];
    const double theta = coordinates[2];
    const double k0 = tangent[0];
    const double k1 = tangent[1];
    const double k2 = tangent[2];
    const double k3 = tangent[3];
    const double two_k2 = 2*k2;
    const double spin2 = pow(spin, 2);
    const double spin2_r = r*spin2;
    const double cos_th = cos(theta);
    const double sin_th = sin(theta);
    const double cos_th_sin_th = cos_th*sin_th;
    const double spin2_r_cos_th_sin_th = cos_th_sin_th*spin2_r;
    const double cos_th_k3 = cos_th*k3;
    const double r2 = pow(r, 2);
    const double r2_plus_spin2 = r2 + spin2;
    const double two_mass = 2*mass;
    const double two_mass_r = r*two_mass;
    const double delta = r2_plus_spin2 - two_mass_

In [12]:
with tempfile.TemporaryDirectory(prefix='relatipy-candidate-acceleration-') as directory:
    acceleration_library_path = Path(directory)/'libcandidate.so'
    acceleration_command = [compiler, '-std=c99', '-pedantic', '-Wall', '-Wextra', '-Werror',
        '-O3', '-fPIC', '-shared', str(acceleration_source), '-lm', '-o', str(acceleration_library_path)]
    subprocess.run(acceleration_command, check=True, capture_output=True, text=True)
    acceleration_library = ctypes.CDLL(str(acceleration_library_path))
    candidate_acceleration = acceleration_library.rp_candidate_kerr_null_acceleration
    candidate_acceleration.argtypes = [ctypes.c_double, ctypes.c_double, pointer, pointer, pointer]
    candidate_acceleration.restype = None
    native_acceleration_rows = []
    for index, sample in enumerate(samples):
        mass, spin, radius, theta, *tangent = sample
        x_buffer = (ctypes.c_double*4)(0.0, radius, theta, 0.0)
        k_buffer = (ctypes.c_double*4)(*tangent)
        derivative = (ctypes.c_double*4)()
        candidate_acceleration(mass, spin, x_buffer, k_buffer, derivative)
        expected = np.asarray(direct_function(*sample), dtype=float)
        actual = np.ctypeslib.as_array(derivative).copy()
        np.testing.assert_allclose(actual, expected, rtol=3e-12, atol=3e-13)
        native_acceleration_rows.append({'muestra': index, 'M': mass, 'a': spin,
            'r': radius, 'theta': theta, 'max_error_absoluto': float(np.max(np.abs(actual-expected))),
            'error_escalado': float(np.max(np.abs(actual-expected)/(1+np.abs(expected)))),
            'compilación_C99_estricta': 'PASS', 'estado': 'PASS'})
native_acceleration_table = pd.DataFrame(native_acceleration_rows)
native_acceleration_table


,muestra,M,a,r,theta,max_error_absoluto,error_escalado,compilación_C99_estricta,estado
0,0,1.000000,0.500000,8.000000,1.100000,3.469447e-18,3.417406e-18,PASS,PASS
1,1,2.000000,-1.400000,15.000000,0.800000,8.673617e-19,8.644590e-19,PASS,PASS
2,2,0.750000,0.000000,6.000000,1.350000,2.168404e-19,2.166839e-19,PASS,PASS
3,3,1.000000,1.000000,1.050000,0.200000,5.551115e-17,4.816710e-17,PASS,PASS
4,4,1.126053,0.412804,12.488281,1.052671,3.330669e-16,1.857719e-16,PASS,PASS
5,5,1.719601,1.019633,16.189994,2.972996,4.440892e-16,1.535121e-16,PASS,PASS
6,6,0.791918,0.382137,1.711614,0.582289,6.661338e-16,2.524311e-16,PASS,PASS
7,7,1.756845,0.350699,27.467165,2.743256,1.776357e-15,2.856049e-16,PASS,PASS
8,8,0.950492,0.661445,24.540796,0.963245,7.105427e-15,1.777696e-16,PASS,PASS
9,9,0.785864,0.378906,7.146033,2.790662,8.881784e-16,2.071759e-16,PASS,PASS


## 5.8. Comparación contra el RHS de producción

La referencia es `rp_kerr_null_geodesic_rhs`, que contrae la conexión **corregida** y no exige norma nula. Compilamos seis unidades C productivas y el texto `c_acceleration` emitido en §5.7 en bibliotecas dentro de `tempfile.TemporaryDirectory`. El arnés se incluye íntegramente en las celdas; no importa archivos de una sesión. Ambos caminos usan `-std=c11 -O3 -fPIC -shared`, sin `fast-math` ni LTO; el candidato también supera el control C99 estricto.

La malla determinista usa $M=1$, $a/M\in\{0,0.5,0.9,0.99,1\}$, radios $r_+(1+10^{-3})$, $r_++0.1$, $6$, $10$, $10^3$ y $10^5$, y $\theta\in\{\pi/2,1,0.1,10^{-4},10^{-8},\pi-10^{-8}\}$. Todos están fuera del horizonte y de la guarda polar `64*DBL_EPSILON`. En cada punto construimos una tangente temporal normalizada con la métrica productiva y dos intentos nulos con $k^t=1$: dirección aleatoria y dirección construida con esa métrica. Si una dirección no admite raíz positiva, se registra como omisión; no se fabrica una tangente. Si hay dos raíces positivas, se toma la menor solo para muestrear tangentes nulas de prueba del RHS (que no depende de la rama); no es la política de condiciones iniciales, que §6.13 fija como rechazo. La ambigüedad se analiza en §5.10.

Para cada aceleración calculamos dos medidas, conservando también los cuatro componentes y el error absoluto:

$$\epsilon_{\mathrm{comp}}=\max_\mu\frac{|a^\mu_{\mathrm{cand}}-a^\mu_{\mathrm{prod}}|}{\max(|a^\mu_{\mathrm{prod}}|,\mathrm{DBL\_MIN})},\qquad
\epsilon_{\mathrm{norm}}=\frac{\|a_{\mathrm{cand}}-a_{\mathrm{prod}}\|_2}{\max(\|a_{\mathrm{prod}}\|_2,\mathrm{DBL\_MIN})}.$$

El denominador por componente puede amplificar diferencias en componentes casi nulas. El resumen por `(spin, tangent_type)` conserva todos los máximos, omisiones y errores. Se muestran las peores filas y una tabla separada de todos los casos omitidos o rechazados. La meta experimental es $10^{-12}$; un exceso se muestra como **FAIL**, se investiga con 80 dígitos y no impide conservar el cuaderno completo. Esto distingue ejecución correcta de cumplimiento numérico. **C-RHS** (cálculo propio; fuentes **S-RHS** = seis archivos identificados por hash en la celda final, `rp_kerr_null_geodesic_rhs` y funciones físicas corregidas; localizador adicional: `case_id` en `comparison_cases`) identifica esta comparación. Revisión humana `pending`.


In [13]:
import ctypes as ct
from datetime import datetime, timezone

SEED = 20260930
FLAGS = ("-std=c11", "-O3", "-fPIC", "-shared")
STATUS = {
    0: "OK", 1: "NULL_POINTER", 2: "NONFINITE_INPUT", 3: "INVALID_PARAMETER",
    4: "COORDINATE_SINGULARITY", 5: "PHYSICAL_SINGULARITY", 6: "NUMERICAL_RANGE",
    7: "NON_TIMELIKE_VELOCITY", 8: "NO_REAL_NULL_TANGENT",
}
WARNING = "Una medición no es garantía de rendimiento ni de precisión en todo el dominio."
_PTR = ct.POINTER(ct.c_double)


def _repo_root(value=None) -> Path:
    """Resolve an explicit root, or discover it from the current directory."""
    start = Path(value).expanduser().resolve() if value is not None else Path.cwd()
    for parent in (start, *start.parents):
        if (parent / "relatipy/native/include/relatipy/kerr_geodesic.h").is_file():
            return parent
        if (parent / "native/include/relatipy/kerr_geodesic.h").is_file():
            return parent.parent
    raise FileNotFoundError("Pass repo_root containing relatipy/native")


def _production_sources(repo_root) -> list[Path]:
    """Six translation units close the RHS/metric symbol dependency graph.

    Orchestrators require their physical evaluators and tensor helpers; both
    physical evaluators require numeric predicates. No integrator is needed.
    Linking with -z defs checks that this graph has no unresolved symbols.
    """
    native = _repo_root(repo_root) / "relatipy/native/src"
    names = ("geodesic/kerr.c", "geodesic/physic/kerr.c", "metric/kerr.c",
             "metric/physic/kerr.c", "utils/numeric.c", "utils/tensor.c")
    sources = [native / name for name in names]
    for source in sources:
        if not source.is_file():
            raise FileNotFoundError(source)
    return sources


def _run(command):
    return subprocess.run([str(x) for x in command], check=True, text=True,
                          capture_output=True).stdout.strip()


def _compile(sources, out_dir, name, repo_root=None) -> Path:
    out = Path(out_dir).expanduser().resolve()
    out.mkdir(parents=True, exist_ok=True)
    sources = [Path(s).resolve() for s in sources]
    compiler = _run(["cc", "--version"])
    digest = hashlib.sha256(compiler.encode() + repr(FLAGS).encode())
    for source in sources:
        digest.update(str(source).encode())
        digest.update(source.read_bytes())
    # Header changes also affect builds and must invalidate the loaded filename.
    if repo_root is not None:
        native = _repo_root(repo_root) / "relatipy/native"
        for header in sorted(native.rglob("*.h")):
            digest.update(str(header.relative_to(native)).encode())
            digest.update(header.read_bytes())
    library = out / f"{name}_{digest.hexdigest()[:16]}.so"
    command = ["cc", *FLAGS]
    if repo_root is not None:
        native = _repo_root(repo_root) / "relatipy/native"
        command += ["-I", native / "include", "-I", native / "src"]
    command += [*sources, "-Wl,-z,defs", "-lm", "-o", library]
    _run(command)
    manifest = {
        "command": [str(x) for x in command], "compiler": compiler,
        "source_sha256": {str(s): hashlib.sha256(s.read_bytes()).hexdigest()
                          for s in sources},
        "utc": datetime.now(timezone.utc).isoformat(),
        "semantics": "C11 -O3; no fast-math, no LTO",
    }
    library.with_suffix(".build.json").write_text(json.dumps(manifest, indent=2))
    return library


def build_production_library(repo_root, out_dir) -> Path:
    """Build the minimal production metric/RHS shared library outside the repo."""
    return _compile(_production_sources(repo_root), out_dir, "production", repo_root)


def build_candidate_library(c_source: str, out_dir) -> Path:
    """Validate strict C99, then build C11 -O3 candidate with no fast-math."""
    out = Path(out_dir).expanduser().resolve()
    out.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(c_source.encode()).hexdigest()[:16]
    source = out / f"candidate_{digest}.c"
    source.write_text(c_source)
    _run(["cc", "-std=c99", "-pedantic", "-Wall", "-Wextra", "-Werror",
          "-fsyntax-only", source])
    return _compile([source], out, "candidate")


def _load(library):
    return library if isinstance(library, ct.CDLL) else ct.CDLL(str(Path(library).resolve()))


def _production(library):
    lib = _load(library)
    lib.rp_kerr_metric.argtypes = [ct.c_double, ct.c_double, _PTR, _PTR]
    lib.rp_kerr_metric.restype = ct.c_int
    lib.rp_kerr_null_geodesic_rhs.argtypes = [ct.c_double, ct.c_double] + [_PTR] * 4
    lib.rp_kerr_null_geodesic_rhs.restype = ct.c_int
    return lib


def _ptr(array):
    return array.ctypes.data_as(_PTR)


def _metric(lib, mass, spin, coordinates):
    x = np.ascontiguousarray(coordinates, dtype=np.float64)
    metric = np.zeros((4, 4), dtype=np.float64)
    status = lib.rp_kerr_metric(mass, spin, _ptr(x), _ptr(metric))
    return status, metric


def _positive_roots(a, b, c):
    """Stable quadratic roots; smallest positive root resolves two-root cases.

    Both positive null roots with k^t=1 outside the horizon are future directed.
    Selecting the smaller one is a documented sampling policy, not an API choice.
    Extended precision reduces cancellation in grid construction only.
    """
    a, b, c = map(np.longdouble, (a, b, c))
    if a == 0:
        roots = [] if b == 0 else [-c / b]
    else:
        discriminant = b*b - 4*a*c
        if discriminant < 0:
            return []
        q = -np.longdouble(0.5) * (b + np.copysign(np.sqrt(discriminant), b))
        roots = [-b/(2*a)] if q == 0 else [q/a, c/q]
    return sorted({float(r) for r in roots if np.isfinite(r) and r > 0})


def _norm_residual(g, tangent, target):
    u = np.asarray(tangent, dtype=np.longdouble)
    terms = np.asarray(g, dtype=np.longdouble) * np.outer(u, u)
    residual = np.sum(terms) - target
    scale = max(np.sum(np.abs(terms)), np.longdouble(abs(target)), np.finfo(float).tiny)
    return float(residual), float(abs(residual)/scale)


def _point_cases(lib, spin, r_label, r, theta, rng):
    x = np.array([0.0, r, theta, rng.uniform(-np.pi, np.pi)])
    base = dict(mass=1.0, spin=float(spin), r_label=r_label, r=float(r),
                theta=float(theta), coordinates=x)
    status, g = _metric(lib, 1.0, spin, x)
    if status:
        return [dict(base, tangent_type=kind, tangent=None, metric_status=status,
                     generation_status="METRIC_ERROR", root_count=0)
                for kind in ("timelike", "null")]
    # Metric-only coordinate-time spatial velocities centered on frame dragging.
    # Solve u^t = 1/sqrt(-g((1,v),(1,v))), then u^i = u^t v^i.
    omega = -g[0, 3] / g[3, 3]
    lapse = np.sqrt(-(np.longdouble(g[0, 0])
                      - np.longdouble(g[0, 3])**2/g[3, 3]))
    spatial_scale = np.sqrt(np.diag(g)[1:])
    direction = rng.normal(size=3)
    direction /= np.linalg.norm(direction)
    velocity = np.asarray(float(lapse) * rng.uniform(0.05, 0.3)
                          * direction / spatial_scale, dtype=float)
    velocity[2] += omega
    unit = np.r_[1.0, velocity]
    norm = np.sum(np.asarray(g, dtype=np.longdouble)
                  * np.outer(unit.astype(np.longdouble), unit.astype(np.longdouble)))
    cases = []
    if norm < 0:
        tangent = unit / float(np.sqrt(-norm))
        residual, relative = _norm_residual(g, tangent, -1)
        cases.append(dict(base, tangent_type="timelike", tangent=tangent,
                          spatial_velocity=velocity, metric_status=0,
                          generation_status="OK", root_count=1,
                          constraint_residual=residual, constraint_relative=relative))
    else:
        cases.append(dict(base, tangent_type="timelike", tangent=None,
                          metric_status=0, generation_status="NO_TIMELIKE_TANGENT", root_count=0))
    # One random direction can be rejected in the ergosphere. Keep its omission
    # visible. A second, metric-constructed direction covers these same points.
    direction = rng.normal(size=3) / spatial_scale
    local_direction = rng.normal(size=3)
    local_direction /= np.linalg.norm(local_direction)
    guaranteed = float(lapse) * local_direction / spatial_scale
    guaranteed[2] += omega
    for label, direction in (("random", direction), ("metric_seeded", guaranteed)):
        n = direction.astype(np.longdouble)
        gl = g.astype(np.longdouble)
        roots = _positive_roots(n @ gl[1:, 1:] @ n,
                                2*gl[0, 1:] @ n, gl[0, 0])
        if not roots:
            cases.append(dict(base, tangent_type="null", tangent=None,
                              direction_label=label, metric_status=0, root_count=0,
                              generation_status="NO_REAL_FUTURE_ROOT"))
            continue
        tangent = np.r_[1.0, roots[0] * direction]
        residual, relative = _norm_residual(g, tangent, 0)
        cases.append(dict(base, tangent_type="null", tangent=tangent,
                          direction_label=label, metric_status=0, root_count=len(roots),
                          generation_status="OK", constraint_residual=residual,
                          constraint_relative=relative))
    return cases


def comparison_grid(prod_lib=None, *, repo_root=None, out_dir=None) -> list[dict]:
    """Construct deterministic timelike/null cases with the production metric.

    With no arguments, discover repo_root from cwd and build into a temporary
    directory. Pass prod_lib to reuse a library in notebook cells. Omitted null
    attempts are ledger entries with tangent=None; compare counts them without
    evaluating a fabricated tangent. Spin is a/M, here M=1.
    """
    if prod_lib is None:
        if out_dir is None:
            with tempfile.TemporaryDirectory(prefix="kerr_grid_") as temporary:
                prod = build_production_library(_repo_root(repo_root), temporary)
                return comparison_grid(prod)
        prod_lib = build_production_library(_repo_root(repo_root), out_dir)
    lib = _production(prod_lib)
    rng = np.random.default_rng(SEED)
    cases = []
    guard = 64*np.finfo(float).eps
    for spin in (0.0, 0.5, 0.9, 0.99, 1.0):
        horizon = 1 + np.sqrt(1 - spin*spin)
        radii = (("horizon_relative", horizon*(1+1e-3)),
                 ("horizon_plus_0.1", horizon+0.1), ("r6", 6.0),
                 ("r10", 10.0), ("r1e3", 1e3), ("r1e5", 1e5))
        for label, r in radii:
            for theta in (np.pi/2, 1.0, 0.1, 1e-4, 1e-8, np.pi-1e-8):
                if guard < theta < np.pi-guard:
                    cases += _point_cases(lib, spin, label, r, theta, rng)
    for index, case in enumerate(cases):
        case["case_id"] = index
    return cases


def _scaled_norm(array):
    magnitude = np.max(np.abs(array))
    return 0.0 if magnitude == 0 else float(magnitude * np.linalg.norm(array/magnitude))


def comparison_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Aggregate maxima and all omitted/error counts by spin and tangent type."""
    data = frame.assign(
        evaluated=frame.production_status.notna().astype(int),
        omitted=frame.production_status.isna().astype(int),
        production_errors=(frame.production_status.notna()
                           & frame.production_status.ne(0)).astype(int),
        candidate_errors=(frame.candidate_finite.eq(False)
                          & frame.production_status.eq(0)).astype(int),
    )
    return data.groupby(["spin", "tangent_type"], as_index=False).agg(
        attempts=("case_id", "size"), evaluated=("evaluated", "sum"),
        omitted=("omitted", "sum"), production_errors=("production_errors", "sum"),
        candidate_errors=("candidate_errors", "sum"),
        max_component_relative=("max_component_relative", "max"),
        max_norm_relative=("norm_relative", "max"),
        max_constraint_relative=("constraint_relative", "max"),
    )


def compare(prod_lib, cand_lib, grid) -> pd.DataFrame:
    """Compare accelerations, retaining every production failure and omission.

    Per-component denominator is max(abs(production_i), DBL_MIN). Near-zero
    components can inflate this measure; norm_relative provides a second view.
    Summary is available via comparison_summary(df).
    """
    prod, cand = _production(prod_lib), _load(cand_lib)
    cand.rp_candidate_kerr_null_acceleration.argtypes = [ct.c_double, ct.c_double] + [_PTR]*3
    cand.rp_candidate_kerr_null_acceleration.restype = None
    rows = []
    for index, case in enumerate(grid):
        row = {key: case.get(key) for key in
               ("spin", "r_label", "r", "theta", "tangent_type", "generation_status",
                "metric_status", "root_count", "direction_label", "constraint_residual",
                "constraint_relative")}
        row.update(case_id=case.get("case_id", index), production_status=None,
                   production_status_name="NOT_EVALUATED", candidate_finite=None,
                   max_component_relative=np.nan, norm_relative=np.nan,
                   max_absolute_error=np.nan)
        if case["tangent"] is not None:
            x = np.ascontiguousarray(case["coordinates"], dtype=np.float64)
            u = np.ascontiguousarray(case["tangent"], dtype=np.float64)
            dx, reference, result = (np.zeros(4) for _ in range(3))
            status = prod.rp_kerr_null_geodesic_rhs(case["mass"], case["spin"],
                        _ptr(x), _ptr(u), _ptr(dx), _ptr(reference))
            row.update(production_status=status, production_status_name=STATUS.get(status, str(status)))
            # The raw candidate has no error contract. Do not evaluate it when
            # production rejects the state; retain production status instead.
            if status == 0:
                cand.rp_candidate_kerr_null_acceleration(case["mass"], case["spin"],
                                                        _ptr(x), _ptr(u), _ptr(result))
                finite = bool(np.all(np.isfinite(result)))
                row["candidate_finite"] = finite
                if finite:
                    diff = result-reference
                    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                        row["max_component_relative"] = float(np.max(
                            np.abs(diff)/np.maximum(np.abs(reference), np.finfo(float).tiny)))
                        row["norm_relative"] = (_scaled_norm(diff)
                            / max(_scaled_norm(reference), np.finfo(float).tiny))
                    row["max_absolute_error"] = float(np.max(np.abs(diff)))
                else:
                    row.update(max_component_relative=np.inf, norm_relative=np.inf,
                               max_absolute_error=np.inf)
                for component in range(4):
                    row[f"production_{component}"] = reference[component]
                    row[f"candidate_{component}"] = result[component]
        rows.append(row)
    frame = pd.DataFrame(rows)
    frame.attrs["seed"] = SEED
    frame.attrs["manual_review"] = "pending"
    return frame


def _benchmark_states(prod, n_states):
    """Create distinct, varied valid null states, without repeating grid rows."""
    rng = np.random.default_rng(SEED)
    states = []
    for index in range(n_states):
        spin = (0.0, 0.5, 0.9, 0.99, 1.0)[index % 5]
        horizon = 1+np.sqrt(1-spin*spin)
        radius = horizon + np.exp(rng.uniform(np.log(0.01), np.log(1e5)))
        theta = rng.uniform(0.02, np.pi-0.02)
        case = _point_cases(prod, spin, "benchmark", radius, theta, rng)[-1]
        if case["tangent"] is None:
            raise RuntimeError("Metric-seeded benchmark direction has no root")
        if case["constraint_relative"] > 1e-12:
            raise RuntimeError("Benchmark tangent violates null constraint")
        x, u = case["coordinates"], case["tangent"]
        dx, du = np.zeros(4), np.zeros(4)
        status = prod.rp_kerr_null_geodesic_rhs(1., spin, _ptr(x), _ptr(u), _ptr(dx), _ptr(du))
        if status:
            raise RuntimeError(f"Benchmark state {index}: production status {status}")
        states.append(np.r_[1., spin, x, u])
    return np.ascontiguousarray(states, dtype=np.float64)


def run_benchmark(repo_root, candidate_c_source, out_dir, repetitions=31,
                  n_states=1024, inner_loops=32, *, bench_driver_source=None) -> pd.DataFrame:
    """Time three paths in C, returning median/IQR/min in ns/evaluation.

    Python creates n_states distinct null states and crosses ctypes once for
    the complete experiment. All three paths use the same states and C11 -O3,
    separate translation units, no LTO and no fast-math. Pass bench_driver_source
    (C text) when pasting this function into a notebook; no se lee un driver externo. Writes only beneath out_dir.
    """
    for name, value in (("repetitions", repetitions), ("n_states", n_states),
                        ("inner_loops", inner_loops)):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"{name} must be a positive integer")
    out = Path(out_dir).expanduser().resolve()
    prod_path = build_production_library(repo_root, out)
    candidate_path = build_candidate_library(candidate_c_source, out)
    source = out / f"candidate_{hashlib.sha256(candidate_c_source.encode()).hexdigest()[:16]}.c"
    if bench_driver_source is None:
        raise ValueError("Se requiere el texto C incrustado bench_driver_source")
    driver = out / "bench_driver.c"
    driver.write_text(bench_driver_source)
    library = _compile([*_production_sources(repo_root), source, driver], out, "benchmark", repo_root)
    prod = _production(prod_path)
    states = _benchmark_states(prod, n_states)
    # Validate both candidates before timing; failures cannot become speedups.
    cases = [dict(mass=s[0], spin=s[1], coordinates=s[2:6], tangent=s[6:10],
                  r_label="benchmark", r=s[3], theta=s[4], tangent_type="null",
                  generation_status="OK") for s in states]
    validation = compare(prod, candidate_path, cases)
    if not validation.candidate_finite.all() or validation.norm_relative.max() > 1e-10:
        raise RuntimeError("Candidate fails pre-benchmark numerical validation (norm tolerance 1e-10)")
    bench = _load(library)
    bench.rp_bench_run.argtypes = [_PTR, ct.c_size_t, ct.c_size_t, ct.c_size_t, _PTR, _PTR]
    bench.rp_bench_run.restype = ct.c_int
    samples = np.zeros((repetitions, 3))
    checksum = np.zeros(1)
    status = bench.rp_bench_run(_ptr(states), n_states, inner_loops, repetitions,
                               _ptr(samples), _ptr(checksum))
    if status:
        raise RuntimeError(f"Native benchmark failed: {status}")
    compiler = _run(["cc", "--version"])
    cpu = "unavailable"
    try:
        for line in _run(["lscpu"]).splitlines():
            if line.strip().startswith("Model name:"):
                cpu = line.split(":", 1)[1].strip()
                break
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass
    utc = datetime.now(timezone.utc).isoformat()
    rows = []
    for index, path in enumerate(("production_public", "candidate_raw", "candidate_validated")):
        values = samples[:, index]
        p25, median, p75 = np.percentile(values, [25, 50, 75])
        rows.append(dict(path=path, median_ns=float(median), p25_ns=float(p25),
                         p75_ns=float(p75), iqr_ns=float(p75-p25), min_ns=float(values.min()),
                         n_states=n_states, repetitions=repetitions, inner_loops=inner_loops,
                         seed=SEED, compiler=compiler.splitlines()[0], cpu=cpu, utc=utc,
                         flags=" ".join(FLAGS), checksum=float(checksum[0]),
                         validation_max_norm_relative=float(validation.norm_relative.max())))
    result = pd.DataFrame(rows)
    result.attrs.update(raw_samples_ns=samples.tolist(), compiler_version=compiler,
                        warning=WARNING, manual_review="pending", source_sha256=hashlib.sha256(
                            candidate_c_source.encode()).hexdigest())
    states.tofile(out / "benchmark_states.bin")
    np.save(out / "benchmark_samples_ns.npy", samples)
    result.to_csv(out / "benchmark_summary.csv", index=False)
    (out / "benchmark_metadata.json").write_text(json.dumps(result.attrs, indent=2))
    return result



In [14]:
# Bibliotecas y artefactos permanecen en un directorio temporal del kernel.
rhs_temp = tempfile.TemporaryDirectory(prefix="relatipy-rhs-comparison-")
rhs_production_path = build_production_library(ROOT, rhs_temp.name)
rhs_candidate_path = build_candidate_library(c_acceleration, rhs_temp.name)
comparison_states = comparison_grid(rhs_production_path)
comparison_cases = compare(rhs_production_path, rhs_candidate_path, comparison_states)
comparison_cases.attrs.clear()  # Facilita selección/concatenación en pandas.
comparison_by_spin = comparison_summary(comparison_cases)
rhs_max_component_relative = float(comparison_cases.max_component_relative.max())
rhs_max_norm_relative = float(comparison_cases.norm_relative.max())
rhs_by_type = comparison_cases.groupby("tangent_type").agg(
    max_component_relative=("max_component_relative", "max"),
    max_norm_relative=("norm_relative", "max"))
rhs_numeric_status = "PASS" if max(rhs_max_component_relative, rhs_max_norm_relative) <= 1e-12 else "FAIL"
# El orden inverso conserva los casos con mayor discrepancia sin ocultar errores.
comparison_cases.sort_values("max_component_relative", ascending=False).head(20)


,spin,r_label,r,theta,tangent_type,generation_status,metric_status,root_count,direction_label,constraint_residual,...,norm_relative,max_absolute_error,production_0,candidate_0,production_1,candidate_1,production_2,candidate_2,production_3,candidate_3
442,1.00,horizon_relative,1.001000,1.000000e-04,null,OK,0,1,random,6.462349e-25,...,1.727603e-13,1.962810e-13,-0.000884,-0.000884,-5.374233e-11,-5.374233e-11,5.302102e-04,5.302102e-04,-1.271988e+00,-1.271988e+00
446,1.00,horizon_relative,1.001000,1.000000e-08,null,OK,0,1,metric_seeded,-3.877409e-24,...,1.425633e-16,1.490116e-08,0.000856,0.000856,-6.600072e-11,-6.600072e-11,6.565075e+00,6.565075e+00,-1.045231e+08,-1.045231e+08
449,1.00,horizon_relative,1.001000,3.141593e+00,null,OK,0,1,metric_seeded,1.184678e-22,...,3.971417e-16,1.192093e-07,-0.000856,-0.000856,-6.612422e-11,-6.612422e-11,-3.598760e-01,-3.598760e-01,3.001682e+08,3.001682e+08
443,1.00,horizon_relative,1.001000,1.000000e-04,null,OK,0,1,metric_seeded,3.815371e-23,...,2.735233e-14,1.749290e-13,-0.000788,-0.000788,-9.390547e-11,-9.390547e-11,7.761986e-04,7.761986e-04,7.128962e+00,7.128962e+00
437,1.00,horizon_relative,1.001000,1.000000e+00,null,OK,0,2,metric_seeded,-5.421011e-20,...,1.638645e-10,1.722292e-13,0.001051,0.001051,-1.359968e-10,-1.359968e-10,5.506543e-08,5.506543e-08,5.251950e-04,5.251950e-04
435,1.00,horizon_relative,1.001000,1.000000e+00,timelike,OK,0,1,NaN,0.000000e+00,...,5.453410e-10,2.522455e-07,-462.507041,-462.507041,-8.093641e-04,-8.093641e-04,3.009673e-01,3.009673e-01,-2.310701e+02,-2.310701e+02
440,1.00,horizon_relative,1.001000,1.000000e-01,null,OK,0,2,metric_seeded,2.541099e-21,...,2.060136e-10,1.433084e-13,0.000700,0.000700,-1.349053e-10,-1.349053e-10,8.851704e-07,8.851704e-07,3.381973e-04,3.381973e-04
434,1.00,horizon_relative,1.001000,1.570796e+00,null,OK,0,2,metric_seeded,-1.084202e-19,...,2.219773e-10,2.422977e-13,0.001092,0.001092,-5.407845e-10,-5.407845e-10,-4.517431e-11,-4.517431e-11,5.454045e-04,5.454045e-04
438,1.00,horizon_relative,1.001000,1.000000e-01,timelike,OK,0,1,NaN,0.000000e+00,...,2.316369e-10,8.328220e-08,-359.845992,-359.845992,-5.065792e-04,-5.065792e-04,1.157263e-01,1.157263e-01,-1.789441e+02,-1.789441e+02
432,1.00,horizon_relative,1.001000,1.570796e+00,timelike,OK,0,1,NaN,0.000000e+00,...,2.223068e-10,1.721777e-07,-774.437767,-774.437766,-7.442752e-04,-7.442752e-04,4.818216e-05,4.818216e-05,-3.870549e+02,-3.870549e+02


In [15]:
comparison_by_spin


,spin,tangent_type,attempts,evaluated,omitted,production_errors,candidate_errors,max_component_relative,max_norm_relative,max_constraint_relative
0,0.00,null,72,72,0,0,0,5.023792e-14,5.515584e-16,1.573423e-16
1,0.00,timelike,36,36,0,0,0,5.017967e-14,3.780856e-16,2.717754e-16
2,0.50,null,72,70,2,0,0,1.296972e-14,5.214754e-15,1.346959e-16
3,0.50,timelike,36,36,0,0,0,4.999172e-14,1.955353e-15,2.133479e-16
4,0.90,null,72,68,4,0,0,6.440754e-13,1.915366e-15,1.487163e-16
5,0.90,timelike,36,36,0,0,0,1.174269e-12,2.662462e-15,2.482622e-16
6,0.99,null,72,67,5,0,0,3.134353e-11,6.853997e-14,1.500024e-16
7,0.99,timelike,36,36,0,0,0,2.265835e-12,4.623858e-13,2.509331e-16
8,1.00,null,72,67,5,0,0,1.839316e-09,2.219773e-10,1.204282e-16
9,1.00,timelike,36,36,0,0,0,6.549091e-10,5.453410e-10,2.925852e-16


In [16]:
comparison_issues = comparison_cases.loc[
    comparison_cases.production_status.isna()
    | comparison_cases.production_status.ne(0)
    | comparison_cases.candidate_finite.eq(False)]
# Las omisiones y status != OK se conservan íntegros, con case_id y motivo.
comparison_issues


,spin,r_label,r,theta,tangent_type,generation_status,metric_status,root_count,direction_label,constraint_residual,...,norm_relative,max_absolute_error,production_0,candidate_0,production_1,candidate_1,production_2,candidate_2,production_3,candidate_3
109,0.50,horizon_relative,1.867891,1.570796,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
112,0.50,horizon_relative,1.867891,1.000000,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
217,0.90,horizon_relative,1.437326,1.570796,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
220,0.90,horizon_relative,1.437326,1.000000,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223,0.90,horizon_relative,1.437326,0.100000,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
235,0.90,horizon_plus_0.1,1.535890,1.570796,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
325,0.99,horizon_relative,1.142208,1.570796,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
328,0.99,horizon_relative,1.142208,1.000000,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
331,0.99,horizon_relative,1.142208,0.100000,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
343,0.99,horizon_plus_0.1,1.241067,1.570796,null,NO_REAL_FUTURE_ROOT,0,0,random,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
if comparison_cases.production_status.dropna().ne(0).any():
    raise ArithmeticError("La malla produjo status != OK; véase comparison_issues")
if not comparison_cases.loc[comparison_cases.production_status.eq(0), "candidate_finite"].all():
    raise ArithmeticError("El candidato produjo valores no finitos; véase comparison_issues")
pd.DataFrame([{
    "validación": "ejecución 5.8 / finitud / status", "resultado": "PASS",
    "intentos": len(comparison_cases), "evaluados": int(comparison_cases.production_status.notna().sum()),
    "omitidos": int(comparison_cases.production_status.isna().sum()),
    "max_error_componente": rhs_max_component_relative, "max_error_normado": rhs_max_norm_relative,
}, {"validación": "meta numérica <= 1e-12 (ambas medidas)", "resultado": rhs_numeric_status,
    "max_error_componente": rhs_max_component_relative, "max_error_normado": rhs_max_norm_relative}])


,validación,resultado,intentos,evaluados,omitidos,max_error_componente,max_error_normado
0,ejecución 5.8 / finitud / status,PASS,540.0,524.0,16.0,1.839316e-09,5.453410e-10
1,meta numérica <= 1e-12 (ambas medidas),FAIL,NaN,NaN,NaN,1.839316e-09,5.453410e-10


### Diagnóstico del error mayor que $10^{-12}$

Las expresiones son simbólicamente equivalentes, pero CSE cambia asociaciones y expande polinomios. Cerca del horizonte extremo, $\Delta$ es pequeño: la emisión calcula `(r²+a²)-2Mr`, mientras producción prepara `(r²-2Mr)+a²`. La resta pierde dígitos y el RHS contiene inversos de $\Delta$. Otras cancelaciones aparecen en la contracción productiva y en polinomios del candidato. El siguiente control compara los dos caminos con la contracción simbólica a **80 dígitos**, usando exactamente los mismos números binarios de entrada. No normaliza de nuevo las tangentes y no cambia el candidato para ajustar la comparación. Se examinan los peores casos por componente y por norma; las cifras reales permiten distinguir error de referencia de error del candidato. **C-RHS-DIAG** (cálculo propio; C-GAMMA §§5.3–5.6, S-RHS, `case_id` de la tabla siguiente) sigue pendiente de revisión manual.


In [18]:
# Referencia de 80 dígitos en las mismas entradas float64, no valores decimales aproximados.
worst_component_id = int(comparison_cases.loc[comparison_cases.max_component_relative.idxmax(), "case_id"])
worst_norm_id = int(comparison_cases.loc[comparison_cases.norm_relative.idxmax(), "case_id"])
diagnostic_rows = []
for case_id in sorted({worst_component_id, worst_norm_id}):
    case = comparison_states[case_id]
    row = comparison_cases.loc[comparison_cases.case_id.eq(case_id)].iloc[0]
    substitutions = {symbol: sp.Rational(float(value)) for symbol, value in zip(
        parameters, (case["mass"], case["spin"], case["r"], case["theta"], *case["tangent"]))}
    reference_80 = [expression.evalf(80, subs=substitutions) for expression in symmetric_acceleration]
    radius, spin, mass = case["r"], case["spin"], case["mass"]
    delta_production = radius*radius-2.0*mass*radius+spin*spin
    delta_candidate = (radius*radius+spin*spin)-2.0*mass*radius
    delta_80 = delta.evalf(80, subs=substitutions)
    diagnostic_rows.append({"case_id": case_id, "spin": spin, "tipo": case["tangent_type"],
        "r": radius, "theta": case["theta"], "cantidad": "delta", "referencia_80": str(delta_80),
        "producción": delta_production, "candidato": delta_candidate,
        "error_rel_producción_80": float(abs((sp.Rational(delta_production)-delta_80)/delta_80)),
        "error_rel_candidato_80": float(abs((sp.Rational(delta_candidate)-delta_80)/delta_80))})
    for component, expected in enumerate(reference_80):
        actual_production = float(row[f"production_{component}"])
        actual_candidate = float(row[f"candidate_{component}"])
        denominator = max(abs(expected), sp.Float(np.finfo(float).tiny, 80))
        diagnostic_rows.append({"case_id": case_id, "spin": spin, "tipo": case["tangent_type"],
            "r": radius, "theta": case["theta"], "cantidad": f"aceleración_{component}",
            "referencia_80": str(expected), "producción": actual_production, "candidato": actual_candidate,
            "error_rel_producción_80": float(abs(sp.Rational(actual_production)-expected)/denominator),
            "error_rel_candidato_80": float(abs(sp.Rational(actual_candidate)-expected)/denominator)})
rhs_precision_diagnostic = pd.DataFrame(diagnostic_rows)
rhs_precision_diagnostic


,case_id,spin,tipo,r,theta,cantidad,referencia_80,producción,candidato,error_rel_producción_80,error_rel_candidato_80
0,435,1.0,timelike,1.001,1.0000,delta,0.00000099999999999977973175191438107190523015...,1.000000e-06,1.000000e-06,8.204637e-11,1.399982e-10
1,435,1.0,timelike,1.001,1.0000,aceleración_0,-462.50704126834936657771277579777639969795026...,-4.625070e+02,-4.625070e+02,4.838233e-10,6.156395e-11
2,435,1.0,timelike,1.001,1.0000,aceleración_1,-0.0008093641172437570582081236810484548866013...,-8.093641e-04,-8.093641e-04,8.215776e-11,1.395086e-10
3,435,1.0,timelike,1.001,1.0000,aceleración_2,0.30096728708371868003548070825887054329969632...,3.009673e-01,3.009673e-01,1.661336e-10,4.887755e-10
4,435,1.0,timelike,1.001,1.0000,aceleración_3,-231.07005189396569443831641296274783977827794...,-2.310701e+02,-2.310701e+02,4.837399e-10,6.141550e-11
5,442,1.0,null,1.001,0.0001,delta,0.00000099999999999977973175191438107190523015...,1.000000e-06,1.000000e-06,8.204637e-11,1.399982e-10
6,442,1.0,null,1.001,0.0001,aceleración_0,-0.0008843793345457948718090429265011840492668...,-8.843793e-04,-8.843793e-04,8.188943e-11,1.400527e-10
7,442,1.0,null,1.001,0.0001,aceleración_1,-0.0000000000537423264352963495821661760228482...,-5.374233e-11,-5.374233e-11,6.802526e-10,1.159064e-09
8,442,1.0,null,1.001,0.0001,aceleración_2,0.00053021024286159218520225811830196803380797...,5.302102e-04,5.302102e-04,2.695474e-17,1.775306e-16
9,442,1.0,null,1.001,0.0001,aceleración_3,-1.2719876276308372657587738333268218295128809...,-1.271988e+00,-1.271988e+00,2.864057e-14,4.904089e-14


## 5.9. Microbenchmark reproducible

El bucle de evaluación y el cronómetro `CLOCK_MONOTONIC` están en C, incrustado a continuación como `bench_driver_source`. Python cruza `ctypes` una sola vez para todas las repeticiones. Cada unidad se compila con `-std=c11 -O3 -fPIC -shared`, sin `fast-math` ni LTO; producción y candidato son unidades separadas. Se calientan los tres caminos y se rota el orden de medición en cada repetición. Un checksum de las cuatro componentes alimenta un destino `volatile` por lote para impedir que se eliminen las evaluaciones.

Comparamos **producción pública**, **candidato crudo** (sin validación) y **candidato con validación equivalente**. La envoltura experimental reproduce punteros, parámetros, finitud, copias, ceros, guardas, estados y comprobación de salida de producción; reutiliza `rp_kerr_geodesic_physic_prepare_state`. Esa preparación y el candidato pueden repetir trigonometría: se mide esta interfaz concreta, no un futuro kernel que reutilice cantidades precalculadas. Ninguna API productiva cambia.

La distribución contiene `n_states=1024` estados nulos distintos, $M=1$, los cinco spins alternados, distancia al horizonte loguniforme entre $0.01$ y $10^5$, $\theta$ uniforme en $[0.02,\pi-0.02]$ y semilla `20260930`. Antes de medir se comprueba la norma nula y se compara cada aceleración candidata con producción (tolerancia normada `1e-10`, control operacional separado de la meta `1e-12` de §5.8). Se ejecutan `31` repeticiones y `32` bucles internos. Las tablas declaran compilador, flags, CPU, fecha UTC, tamaño y estadísticas en **ns/evaluación**: mediana, p25, p75, IQR y mínimo. La tabla de muestras conserva todas las repeticiones.

**Una medición no es garantía** de rendimiento ni de precisión en todo el dominio. No se fija afinidad ni frecuencia de CPU; carga concurrente y sistema operativo pueden cambiar las cifras. **C-BENCH** (medición propia; driver incrustado `rp_bench_run`/`evaluate_batch`, semilla y filas por camino de `benchmark_table`) sigue pendiente de revisión humana.


In [19]:
bench_driver_source = r"""/**
 * Scratch-only Kerr acceleration microbenchmark; not a production API.
 *
 * Compile every translation unit with -std=c11 -O3, without LTO or fast-math.
 * The candidate remains in a separate translation unit, so calls cannot be
 * folded into the driver. Caller owns all buffers; no allocation or retained
 * pointers. Units and coordinate order match the production Kerr interface.
 */
#define _POSIX_C_SOURCE 200809L

#include "relatipy/kerr_geodesic.h"
#include "geodesic/physic/kerr.h"
#include "utils/tensor.h"

#include <math.h>
#include <stddef.h>
#include <stdint.h>
#include <time.h>

void rp_candidate_kerr_null_acceleration(
    double mass,
    double spin,
    const double coordinates[4],
    const double tangent[4],
    double tangent_derivative[4]
);

/**
 * Apply exactly the production RHS validation/copy/zero conventions around
 * the raw candidate. Partial output overlaps are forbidden, as in production.
 * Calling the production preparation function preserves its numeric guards,
 * overflow checks, status precedence, and trigonometric preparation work.
 */
rp_kerr_status rp_bench_candidate_rhs(
    double mass,
    double spin,
    const double coordinates[RP_KERR_DIM],
    const double tangent[RP_KERR_DIM],
    double coordinate_derivative[RP_KERR_DIM],
    double tangent_derivative[RP_KERR_DIM]
)
{
    struct rp_kerr_geodesic_state state;
    rp_kerr_status status = RP_KERR_STATUS_OK;
    size_t index;

    if (coordinate_derivative == NULL || tangent_derivative == NULL) {
        if (coordinate_derivative != NULL) {
            rp_tensor_zero(coordinate_derivative, RP_KERR_DIM);
        }
        if (tangent_derivative != NULL) {
            rp_tensor_zero(tangent_derivative, RP_KERR_DIM);
        }
        return RP_KERR_STATUS_NULL_POINTER;
    }
    if (coordinate_derivative == tangent_derivative) {
        rp_tensor_zero(coordinate_derivative, RP_KERR_DIM);
        return RP_KERR_STATUS_INVALID_PARAMETER;
    }
    if (coordinates == NULL || tangent == NULL) {
        rp_tensor_zero(coordinate_derivative, RP_KERR_DIM);
        rp_tensor_zero(tangent_derivative, RP_KERR_DIM);
        return RP_KERR_STATUS_NULL_POINTER;
    }

    if (!isfinite(mass) || !isfinite(spin)) {
        status = RP_KERR_STATUS_NONFINITE_INPUT;
    } else {
        for (index = 0U; index < RP_KERR_DIM; ++index) {
            if (!isfinite(coordinates[index]) || !isfinite(tangent[index])) {
                status = RP_KERR_STATUS_NONFINITE_INPUT;
                break;
            }
            state.x[index] = coordinates[index];
            state.u[index] = tangent[index];
        }
        if (status == RP_KERR_STATUS_OK) {
            if (!(mass > 0.0) || fabs(spin) > mass) {
                status = RP_KERR_STATUS_INVALID_PARAMETER;
            } else {
                status = rp_kerr_geodesic_physic_prepare_state(
                    mass, spin, &state
                );
            }
        }
    }
    rp_tensor_zero(coordinate_derivative, RP_KERR_DIM);
    rp_tensor_zero(tangent_derivative, RP_KERR_DIM);
    if (status != RP_KERR_STATUS_OK) {
        return status;
    }

    for (index = 0U; index < RP_KERR_DIM; ++index) {
        coordinate_derivative[index] = state.u[index];
    }
    rp_candidate_kerr_null_acceleration(
        mass, spin, state.x, state.u, tangent_derivative
    );
    if (!rp_tensor_is_finite(coordinate_derivative, RP_KERR_DIM)
        || !rp_tensor_is_finite(tangent_derivative, RP_KERR_DIM)) {
        rp_tensor_zero(coordinate_derivative, RP_KERR_DIM);
        rp_tensor_zero(tangent_derivative, RP_KERR_DIM);
        return RP_KERR_STATUS_NUMERICAL_RANGE;
    }
    return RP_KERR_STATUS_OK;
}

/* Route dispatch occurs outside the evaluation loops. Accumulate all four
 * acceleration components equally for each route, then publish to a volatile
 * sink once per batch. This avoids imposing a volatile store on every call. */
static int evaluate_batch(
    int route,
    const double *states,
    size_t n_states,
    size_t inner_loops,
    volatile double *sink
)
{
    double dx[RP_KERR_DIM];
    double du[RP_KERR_DIM];
    double sum = 0.0;
    size_t loop;
    size_t index;

    if (route == 0) {
        for (loop = 0U; loop < inner_loops; ++loop) {
            for (index = 0U; index < n_states; ++index) {
                const double *row = states + 10U * index;
                if (rp_kerr_null_geodesic_rhs(
                    row[0], row[1], row + 2U, row + 6U, dx, du
                ) != RP_KERR_STATUS_OK) {
                    return 4;
                }
                sum += du[0] + 0.5 * du[1] + 0.25 * du[2] + 0.125 * du[3];
            }
        }
    } else if (route == 1) {
        for (loop = 0U; loop < inner_loops; ++loop) {
            for (index = 0U; index < n_states; ++index) {
                const double *row = states + 10U * index;
                rp_candidate_kerr_null_acceleration(
                    row[0], row[1], row + 2U, row + 6U, du
                );
                sum += du[0] + 0.5 * du[1] + 0.25 * du[2] + 0.125 * du[3];
            }
        }
    } else {
        for (loop = 0U; loop < inner_loops; ++loop) {
            for (index = 0U; index < n_states; ++index) {
                const double *row = states + 10U * index;
                if (rp_bench_candidate_rhs(
                    row[0], row[1], row + 2U, row + 6U, dx, du
                ) != RP_KERR_STATUS_OK) {
                    return 4;
                }
                sum += du[0] + 0.5 * du[1] + 0.25 * du[2] + 0.125 * du[3];
            }
        }
    }
    *sink = sum;
    return isfinite(sum) ? 0 : 5;
}

/**
 * Time three routes using CLOCK_MONOTONIC, after checking every state and
 * warming up every route. Route order rotates for each repetition.
 *
 * states: n_states contiguous rows [M, a, t, r, theta, phi, u0, u1, u2, u3].
 * ns_per_eval: caller-owned repetitions-by-3 contiguous output, columns
 *             production public RHS, raw candidate, validated candidate.
 * checksum: caller-owned scalar output, receives accumulated volatile sinks.
 * Buffers must not overlap. Outputs are valid only on return value zero.
 * Return 1 invalid pointer/count/overflow, 2 rejected state, 3 clock failure,
 *        4 timed evaluation failure, 5 nonfinite checksum or invalid duration.
 * An individual timing measurement does not guarantee future performance.
 */
int rp_bench_run(
    const double *states,
    size_t n_states,
    size_t inner_loops,
    size_t repetitions,
    double *ns_per_eval,
    double *checksum
)
{
    volatile double sink = 0.0;
    double accumulated = 0.0;
    double dx[RP_KERR_DIM];
    double du[RP_KERR_DIM];
    size_t index;
    size_t repetition;
    size_t slot;
    int route;

    if (states == NULL || ns_per_eval == NULL || checksum == NULL
        || n_states == 0U || inner_loops == 0U || repetitions == 0U
        || n_states > SIZE_MAX / (10U * sizeof(double))
        || repetitions > SIZE_MAX / (3U * sizeof(double))
        || inner_loops > SIZE_MAX / n_states) {
        return 1;
    }
    *checksum = 0.0;
    for (index = 0U; index < 3U * repetitions; ++index) {
        ns_per_eval[index] = 0.0;
    }
    /* Untimed preflight verifies all states accepted by both wrappers and
     * all raw candidate outputs finite, without adding checks to its timer. */
    for (index = 0U; index < n_states; ++index) {
        const double *row = states + 10U * index;
        if (rp_kerr_null_geodesic_rhs(
            row[0], row[1], row + 2U, row + 6U, dx, du
        ) != RP_KERR_STATUS_OK
            || rp_bench_candidate_rhs(
                row[0], row[1], row + 2U, row + 6U, dx, du
            ) != RP_KERR_STATUS_OK) {
            return 2;
        }
        rp_candidate_kerr_null_acceleration(
            row[0], row[1], row + 2U, row + 6U, du
        );
        if (!rp_tensor_is_finite(du, RP_KERR_DIM)) {
            return 2;
        }
    }
    for (route = 0; route < 3; ++route) {
        const int status = evaluate_batch(
            route, states, n_states, 1U, &sink
        );
        if (status != 0) {
            return status;
        }
        accumulated += sink;
    }
    for (repetition = 0U; repetition < repetitions; ++repetition) {
        for (slot = 0U; slot < 3U; ++slot) {
            struct timespec start;
            struct timespec end;
            double elapsed_ns;
            int status;

            route = (int)((repetition + slot) % 3U);
            if (clock_gettime(CLOCK_MONOTONIC, &start) != 0) {
                return 3;
            }
            status = evaluate_batch(
                route, states, n_states, inner_loops, &sink
            );
            if (clock_gettime(CLOCK_MONOTONIC, &end) != 0) {
                return 3;
            }
            if (status != 0) {
                return status;
            }
            elapsed_ns = (double)(end.tv_sec - start.tv_sec) * 1.0e9
                + (double)(end.tv_nsec - start.tv_nsec);
            if (!(elapsed_ns > 0.0) || !isfinite(elapsed_ns)) {
                return 5;
            }
            ns_per_eval[3U * repetition + (size_t)route] = elapsed_ns
                / ((double)n_states * (double)inner_loops);
            accumulated += sink;
        }
    }
    if (!isfinite(accumulated)) {
        return 5;
    }
    *checksum = accumulated;
    return 0;
}
"""


In [20]:
benchmark_temp = tempfile.TemporaryDirectory(prefix="relatipy-native-benchmark-")
benchmark_table = run_benchmark(ROOT, c_acceleration, benchmark_temp.name,
    repetitions=31, n_states=1024, inner_loops=32, bench_driver_source=bench_driver_source)
benchmark_statistics = benchmark_table[["path", "median_ns", "p25_ns", "p75_ns", "iqr_ns", "min_ns"]].copy()
benchmark_statistics


,path,median_ns,p25_ns,p75_ns,iqr_ns,min_ns
0,production_public,116.173798,116.024170,116.540527,0.516357,115.551697
1,candidate_raw,144.435181,144.188538,144.915985,0.727448,143.724274
2,candidate_validated,167.329926,167.179138,167.643707,0.464569,166.924713


In [21]:
benchmark_environment = benchmark_table[["compiler", "cpu", "utc", "flags", "n_states",
    "repetitions", "inner_loops", "seed", "checksum", "validation_max_norm_relative"]].drop_duplicates()
benchmark_environment


,compiler,cpu,utc,flags,n_states,repetitions,inner_loops,seed,checksum,validation_max_norm_relative
0,cc (GCC) 16.2.1 20260810,AMD Ryzen AI 7 350 w/ Radeon 860M,2026-10-01T01:48:02.250558+00:00,-std=c11 -O3 -fPIC -shared,1024,31,32,20260930,7965.884525,2.758646e-12


In [22]:
benchmark_samples = pd.DataFrame(benchmark_table.attrs["raw_samples_ns"],
    columns=["production_public", "candidate_raw", "candidate_validated"])
benchmark_samples.index.name = "repetición"
benchmark_samples


,production_public,candidate_raw,candidate_validated
repetición,,,
0,116.173798,145.138489,168.812042
1,115.820221,145.052246,168.616638
2,120.976776,144.133179,167.269989
3,116.033722,144.298615,168.445679
4,117.690460,143.773468,167.050690
5,116.924927,143.724274,166.924713
6,116.130676,143.973206,167.377960
7,116.782104,144.040192,167.156860
8,116.016876,143.978729,167.219849


## 5.10. Rama nula de la cuadrivelocidad desde velocidad coordenada

La sección 6.13 de `arquitectura/arquitectura.md` fija la dirección coordenada y la escala inicial $k^t=1$. No usa la norma temporal $-1$: escribimos $k=(1,s n^r,s n^\theta,s n^\phi)$, con $s>0$, y resolvemos

$$A s^2+B s+C=0,\qquad A=g_{rr}(n^r)^2+g_{\theta\theta}(n^\theta)^2+g_{\phi\phi}(n^\phi)^2,\quad B=2g_{t\phi}n^\phi,\quad C=g_{tt}.$$

Aquí $n^i$ fija una semirrecta espacial, no una velocidad física medida por un observador. Los componentes de $n$ son coordenados y no comparten necesariamente unidades. Las siguientes conclusiones son una derivación del experimento, pendiente de revisión manual:

- En el exterior del horizonte, $\Delta>0$, $\Sigma>0$ y $0<\theta<\pi$; el bloque espacial diagonal es positivo y $A>0$ para toda dirección no nula.
- Fuera de la ergosfera ($C<0$), el producto de raíces $C/A<0$ garantiza exactamente una raíz positiva y una negativa.
- Dentro de la ergosfera ($C>0$), las raíces reales simples tienen el mismo signo. Hay dos positivas si $D=B^2-4AC>0$ y $B<0$; en los demás casos no hay positivas. En la frontera $D=0$, $B<0$, hay una raíz positiva doble. Por tanto, «cero o dos» cuenta multiplicidades y requiere explicitar ese caso límite.
- Sobre el límite estático ($C=0$), las raíces son $0$ y $-B/A$; este candidato exige $s>0$ y descarta la raíz cero.

**Hallazgo para el coordinador:** la exigencia «raíz futura» no elimina la ambigüedad dentro de la ergosfera. En el exterior $g^{tt}<0$, por lo que $dt$ es temporal y ambas raíces positivas con $k^t=1$ son futuras bajo la orientación acordada. No debe reemplazarse ese criterio por $E=-k_t>0$ sin una decisión adicional.

**Contrato aplicable (§6.13, cerrado el 2026-09-30):** una dirección con dos raíces futuras en la ergorregión se **rechaza**; la elección de rama queda **pendiente** (la resolvería la entrada por marco local de observador). El candidato sigue ese contrato: devuelve `0` con una única raíz positiva (incluida la raíz doble $D=0$), `1` sin raíz positiva y `2` con dos raíces positivas distintas, sin escribir una tangente en los casos rechazados. No elige rama. La decisión 7 de §14 sobre código generado y el marco local de observador continúan **PENDIENTES**.


In [23]:
# Derivación simbólica, desde g_sym corregida de §5.2.
s_null = sp.Symbol("s_null", positive=True, real=True)
n_r, n_theta, n_phi = sp.symbols("n_r n_theta n_phi", real=True)
k_null_symbolic = sp.Matrix([1, s_null*n_r, s_null*n_theta, s_null*n_phi])
null_A = g_sym[1,1]*n_r**2 + g_sym[2,2]*n_theta**2 + g_sym[3,3]*n_phi**2
null_B = 2*g_sym[0,3]*n_phi
null_C = g_sym[0,0]
null_polynomial_difference = sp.expand((k_null_symbolic.T*g_sym*k_null_symbolic)[0]
                                      - (null_A*s_null**2 + null_B*s_null + null_C))
if null_polynomial_difference != 0:
    raise ArithmeticError("La contracción nula no coincide con la cuadrática")
null_derivation = pd.DataFrame([
    {"cantidad": "A", "expresión": null_A},
    {"cantidad": "B", "expresión": null_B},
    {"cantidad": "C", "expresión": null_C},
    {"cantidad": "g(k,k) - (A s²+B s+C)", "expresión": null_polynomial_difference},
])
null_derivation


,cantidad,expresión
0,A,n_phi**2*(2*mass*r*spin**2*sin(theta)**2/(r**2...
1,B,-4*mass*n_phi*r*spin*sin(theta)**2/(r**2 + spi...
2,C,2*mass*r/(r**2 + spin**2*cos(theta)**2) - 1
3,"g(k,k) - (A s²+B s+C)",0


Para evitar la cancelación de $-B+\sqrt D$ cuando $B$ y $\sqrt D$ casi coinciden, calculamos

$$q=-\frac12\left(B+\operatorname{copysign}(\sqrt D,B)\right),\qquad s_1=\frac qA,\qquad s_2=\frac Cq.$$

Antes de calcular los coeficientes, C divide la dirección por su máximo componente absoluto. Esta reescala positiva conserva la semirrecta y las tangentes, evita cuadrados innecesariamente grandes y hace reproducible la prueba de magnitudes. La escala final respecto a la dirección original es la raíz calculada dividida por ese máximo.

La segunda raíz procede del producto de raíces. Solo se evalúa cuando $q\ne0$; se filtran raíces finitas y positivas. No se recorta un discriminante negativo a cero: ese caso se devuelve como ausencia de raíz, para no fabricar una dirección permitida. El contrato experimental devuelve `0` al obtener tangente y `1` si no obtiene raíz positiva representable. Las entradas deben pertenecer al dominio exterior válido; estas comprobaciones aritméticas no reemplazan la validación de parámetros, horizonte y guarda polar en la frontera pública. El candidato no reserva memoria, no usa estado global y solo escribe la tangente al obtener una raíz.

Las expresiones de métrica que siguen transcriben `rp_kerr_physic_prepare_point` y `rp_kerr_physic_evaluate_metric` de `relatipy/native/src/metric/physic/kerr.c`, cotejadas simbólicamente en §5.2. No proceden del snapshot legado.


In [24]:
c_null_tangent = r"""#include <math.h>

/* Candidato experimental: BL, firma -+++, G=c=1, M=mass, a=spin.
 * Dominio de entrada validado por el llamador: exterior del horizonte,
 * fuera de la guarda polar, parametros y direccion finitos.
 * Codigos: 0 una raiz positiva (incluida la doble); 1 sin raiz positiva;
 * 2 dos raices positivas distintas, rechazadas como en la seccion 6.13.
 * El llamador posee todos los buffers. Sin asignaciones ni estado global.
 */
int rp_candidate_kerr_null_tangent_from_direction(
    double mass, double spin,
    const double coordinates[4],
    const double direction[3],
    double tangent[4])
{
    const double radius = coordinates[1];
    const double radius_squared = radius * radius;
    const double spin_squared = spin * spin;
    const double sin_theta = sin(coordinates[2]);
    const double cos_theta = cos(coordinates[2]);
    const double sin_squared = sin_theta * sin_theta;
    const double sigma = radius_squared
        + spin_squared * cos_theta * cos_theta;
    const double delta = radius_squared - 2.0 * mass * radius + spin_squared;
    const double common = 2.0 * mass * radius / sigma;
    const double g_tt = -(1.0 - common);
    const double g_tphi = -common * spin * sin_squared;
    const double g_rr = sigma / delta;
    const double g_theta_theta = sigma;
    const double g_phi_phi = (radius_squared + spin_squared
        + common * spin_squared * sin_squared) * sin_squared;
    const double direction_magnitude = fmax(fabs(direction[0]),
        fmax(fabs(direction[1]), fabs(direction[2])));
    const double n_r = direction[0] / direction_magnitude;
    const double n_theta = direction[1] / direction_magnitude;
    const double n_phi = direction[2] / direction_magnitude;
    const double quadratic_a = g_rr * n_r * n_r
        + g_theta_theta * n_theta * n_theta
        + g_phi_phi * n_phi * n_phi;
    const double quadratic_b = 2.0 * g_tphi * n_phi;
    const double quadratic_c = g_tt;
    const double discriminant = quadratic_b * quadratic_b
        - 4.0 * quadratic_a * quadratic_c;
    double q;
    double root_one;
    double root_two;
    double scale = 0.0;
    double spatial_r;
    double spatial_theta;
    double spatial_phi;

    if (!(direction_magnitude > 0.0) || !isfinite(direction_magnitude)
        || !(quadratic_a > 0.0) || !isfinite(quadratic_a)
        || !isfinite(quadratic_b) || !isfinite(quadratic_c)
        || !(discriminant >= 0.0) || !isfinite(discriminant)) {
        return 1;
    }
    q = -0.5 * (quadratic_b + copysign(sqrt(discriminant), quadratic_b));
    root_one = q / quadratic_a;
    root_two = q != 0.0 ? quadratic_c / q : 0.0;
    if (root_one > 0.0 && isfinite(root_one)
        && root_two > 0.0 && isfinite(root_two)
        && discriminant > 0.0) {
        return 2;
    }
    if (root_one > 0.0 && isfinite(root_one)) {
        scale = root_one;
    } else if (root_two > 0.0 && isfinite(root_two)) {
        scale = root_two;
    }
    if (!(scale > 0.0)) {
        return 1;
    }
    spatial_r = scale * n_r;
    spatial_theta = scale * n_theta;
    spatial_phi = scale * n_phi;
    if (!isfinite(spatial_r) || !isfinite(spatial_theta)
        || !isfinite(spatial_phi)) {
        return 1;
    }
    tangent[0] = 1.0;
    tangent[1] = spatial_r;
    tangent[2] = spatial_theta;
    tangent[3] = spatial_phi;
    return 0;
}
"""
# El archivo de compilación es temporal; exportar requiere variable de entorno.
null_source_temp = tempfile.TemporaryDirectory(prefix="relatipy-emitted-null-")
null_source_path = Path(null_source_temp.name) / "candidate_null_tangent.c"
null_source_path.write_text(c_null_tangent, encoding="utf-8")
if EXPORT_DIR is not None:
    (EXPORT_DIR/"candidate_null_tangent.c").write_text(c_null_tangent, encoding="utf-8")
from IPython.display import Code, display
# La celda muestra la función completa, idéntica al archivo temporal.
display(Code(c_null_tangent, language="c"))


#include <math.h>

/* Candidato experimental: BL, firma -+++, G=c=1, M=mass, a=spin.
 * Dominio de entrada validado por el llamador: exterior del horizonte,
 * fuera de la guarda polar, parametros y direccion finitos.
 * Codigos: 0 una raiz positiva (incluida la doble); 1 sin raiz positiva;
 * 2 dos raices positivas distintas, rechazadas como en la seccion 6.13.
 * El llamador posee todos los buffers. Sin asignaciones ni estado global.
 */
int rp_candidate_kerr_null_tangent_from_direction(
    double mass, double spin,
    const double coordinates[4],
    const double direction[3],
    double tangent[4])
{
    const double radius = coordinates[1];
    const double radius_squared = radius * radius;
    const double spin_squared = spin * spin;
    const double sin_theta = sin(coordinates[2]);
    const double cos_theta = cos(coordinates[2]);
    const double sin_squared = sin_theta * sin_theta;
    const double sigma = radius_squared
        + spin_squared * cos_theta * cos_theta;
    const double delta = radius_squared - 2.0 * mass * radius + spin_squared;
    const double common = 2.0 * mass * radius / sigma;
    const double g_tt = -(1.0 - common);
    const double g_tphi = -common * spin * sin_squared;
    const double g_rr = sigma / delta;
    const double g_theta_theta = sigma;
    const double g_phi_phi = (radius_squared + spin_squared
        + common * spin_squared * sin_squared) * sin_squared;
    const double direction_magnitude = fmax(fabs(direction[0]),
        fmax(fabs(direction[1]), fabs(direction[2])));
    const double n_r = direction[0] / direction_magnitude;
    const double n_theta = direction[1] / direction_magnitude;
    const double n_phi = direction[2] / direction_magnitude;
    const double quadratic_a = g_rr * n_r * n_r
        + g_theta_theta * n_theta * n_theta
        + g_phi_phi * n_phi * n_phi;
    const double quadratic_b = 2.0 * g_tphi * n_phi;
    const double quadratic_c = g_tt;
    const double discriminant = quadratic_b * quadratic_b
        - 4.0 * quadratic_a * quadratic_c;
    double q;
    double root_one;
    double root_two;
    double scale = 0.0;
    double spatial_r;
    double spatial_theta;
    double spatial_phi;

    if (!(direction_magnitude > 0.0) || !isfinite(direction_magnitude)
        || !(quadratic_a > 0.0) || !isfinite(quadratic_a)
        || !isfinite(quadratic_b) || !isfinite(quadratic_c)
        || !(discriminant >= 0.0) || !isfinite(discriminant)) {
        return 1;
    }
    q = -0.5 * (quadratic_b + copysign(sqrt(discriminant), quadratic_b));
    root_one = q / quadratic_a;
    root_two = q != 0.0 ? quadratic_c / q : 0.0;
    if (root_one > 0.0 && isfinite(root_one)
        && root_two > 0.0 && isfinite(root_two)
        && discriminant > 0.0) {
        return 2;
    }
    if (root_one > 0.0 && isfinite(root_one)) {
        scale = root_one;
    } else if (root_two > 0.0 && isfinite(root_two)) {
        scale = root_two;
    }
    if (!(scale > 0.0)) {
        return 1;
    }
    spatial_r = scale * n_r;
    spatial_theta = scale * n_theta;
    spatial_phi = scale * n_phi;
    if (!isfinite(spatial_r) || !isfinite(spatial_theta)
        || !isfinite(spatial_phi)) {
        return 1;
    }
    tangent[0] = 1.0;
    tangent[1] = spatial_r;
    tangent[2] = spatial_theta;
    tangent[3] = spatial_phi;
    return 0;
}

In [25]:
# La biblioteca permanece cargada durante la malla; sus archivos son temporales.
null_candidate_temp = tempfile.TemporaryDirectory(prefix="kerr_null_candidate_")
null_library_path = Path(null_candidate_temp.name) / "candidate_null_tangent.so"
null_syntax_command = [compiler, "-std=c99", "-pedantic", "-Wall", "-Wextra", "-Werror",
                       "-fsyntax-only", str(null_source_path)]
null_compile_command = [compiler, "-std=c99", "-pedantic", "-Wall", "-Wextra", "-Werror",
                        "-O3", "-fPIC", "-shared", str(null_source_path), "-lm",
                        "-o", str(null_library_path)]
null_compile_rows = []
for null_label, null_command in [("C99 estricto: sintaxis", null_syntax_command),
                                 ("C99 estricto: biblioteca temporal", null_compile_command)]:
    null_process = subprocess.run(null_command, capture_output=True, text=True, check=True)
    null_compile_rows.append({"validación": null_label, "estado": "PASS",
                              "comando": " ".join(null_command),
                              "stderr": null_process.stderr.strip()})
null_library = ctypes.CDLL(str(null_library_path))
null_pointer = ctypes.POINTER(ctypes.c_double)
null_native_function = null_library.rp_candidate_kerr_null_tangent_from_direction
null_native_function.argtypes = [ctypes.c_double, ctypes.c_double,
                                 null_pointer, null_pointer, null_pointer]
null_native_function.restype = ctypes.c_int

def candidate_null_tangent(mass_value, spin_value, coordinates_value, direction_value):
    """Ejecuta el candidato experimental; el llamador conserva los buffers."""
    coordinates_array = np.ascontiguousarray(coordinates_value, dtype=np.float64)
    direction_array = np.ascontiguousarray(direction_value, dtype=np.float64)
    tangent_array = np.full(4, np.nan, dtype=np.float64)
    status = null_native_function(mass_value, spin_value,
                                 coordinates_array.ctypes.data_as(null_pointer),
                                 direction_array.ctypes.data_as(null_pointer),
                                 tangent_array.ctypes.data_as(null_pointer))
    return status, tangent_array

null_compile_table = pd.DataFrame(null_compile_rows)
null_compile_table


,validación,estado,comando,stderr
0,C99 estricto: sintaxis,PASS,/usr/bin/cc -std=c99 -pedantic -Wall -Wextra -...,
1,C99 estricto: biblioteca temporal,PASS,/usr/bin/cc -std=c99 -pedantic -Wall -Wextra -...,


La malla determinista cubre $a/M\in\{0,0.5,0.9,0.99,1\}$, dos masas geométricas, tres radios (horizonte con margen `2e-6`, intermedio y lejano), ángulos centrales y distancias al eje de `1e-8` y `1e-12` radianes. Estos ángulos superan la guarda `64*DBL_EPSILON`. Cada dirección se prueba con tres magnitudes para comprobar que fija la semirrecta. Incluimos direcciones radiales, polares, azimutales corrotantes/contrarrotantes y mixtas.

La referencia es la métrica **nativa de producción** `rp_kerr_metric` compilada en §5.2. Para cada tangente aceptada medimos

$$\rho=\frac{|g(k,k)|}{|g_{tt}|+|2g_{t\phi}k^\phi|+|g_{rr}(k^r)^2|+|g_{\theta\theta}(k^\theta)^2|+|g_{\phi\phi}(k^\phi)^2|}.$$

Esta escala evita dividir solo por $g_{tt}$, que se anula en el límite estático. Una dirección sin raíz dentro de la ergosfera es un resultado geométrico esperado y se contabiliza por separado. Ningún rechazo se incluye en el máximo de residuos de tangentes válidas. La tabla conserva el número de direcciones ambiguas; no las oculta al elegir una raíz.


In [26]:
# Orden explícito y sin aleatoriedad. Las escalas no cambian la semirrecta.
null_spins = (0.0, 0.5, 0.9, 0.99, 1.0)
null_masses = (1.0, 2.0)
null_theta_values = (1e-12, 1e-8, 0.2, np.pi/2, np.pi-0.2,
                     np.pi-1e-8, np.pi-1e-12)
null_direction_specs = (
    ("radial_saliente", (1.0, 0.0, 0.0)),
    ("radial_entrante", (-1.0, 0.0, 0.0)),
    ("polar_positiva", (0.0, 1.0, 0.0)),
    ("polar_negativa", (0.0, -1.0, 0.0)),
    ("azimutal_corrotante", (0.0, 0.0, 1.0)),
    ("azimutal_contrarrotante", (0.0, 0.0, -1.0)),
    ("mixta_corrotante", (1.0, 0.2, 0.3)),
    ("mixta_contrarrotante", (-1.0, -0.2, -0.3)),
    ("casi_azimutal_corrotante", (1e-6, 0.0, 1.0)),
    ("casi_azimutal_contrarrotante", (1e-6, 0.0, -1.0)),
)
null_direction_scales = (1e-6, 1.0, 1e6)
null_polar_guard = 64*np.finfo(np.float64).eps
null_grid_rows = []
for null_mass in null_masses:
    for null_spin_ratio in null_spins:
        null_spin = null_mass*null_spin_ratio
        null_horizon = null_mass + np.sqrt(null_mass**2-null_spin**2)
        null_radius_specs = (("cerca_horizonte", null_horizon*(1.0+2e-6)),
                             ("intermedio", null_horizon+2.0*null_mass),
                             ("lejano", 50.0*null_mass))
        for null_radius_label, null_radius in null_radius_specs:
            for null_theta in null_theta_values:
                if not (null_polar_guard < null_theta < np.pi-null_polar_guard):
                    raise ValueError("La malla viola la guarda polar")
                null_coordinates = np.array([0.0, null_radius, null_theta, 0.3])
                null_metric = native_metric(null_mass, null_spin, null_coordinates)
                for null_direction_label, null_direction_base in null_direction_specs:
                    null_direction_base = np.array(null_direction_base, dtype=np.float64)
                    null_first_tangent = None
                    for null_direction_scale in null_direction_scales:
                        null_direction = null_direction_scale*null_direction_base
                        null_a_numeric = sum(null_metric[i+1,i+1]*null_direction[i]**2
                                             for i in range(3))
                        null_b_numeric = 2*null_metric[0,3]*null_direction[2]
                        null_c_numeric = null_metric[0,0]
                        null_discriminant_numeric = null_b_numeric**2-4*null_a_numeric*null_c_numeric
                        # Signos/producto permiten clasificar sin raíces inestables.
                        if null_c_numeric < 0:
                            null_positive_root_count = 1
                        elif null_discriminant_numeric >= 0 and null_b_numeric < 0:
                            null_positive_root_count = 1 if null_c_numeric == 0 else 2
                        else:
                            null_positive_root_count = 0
                        null_status, null_tangent = candidate_null_tangent(
                            null_mass, null_spin, null_coordinates, null_direction)
                        # §6.13: una raíz -> 0; ninguna -> 1; dos distintas -> 2 (rechazo).
                        null_expected_status = {1: 0, 0: 1, 2: 2}[null_positive_root_count]
                        if null_status != null_expected_status:
                            raise ArithmeticError("El estado C contradice la clasificación geométrica")
                        null_residual_relative = np.nan
                        null_scale_difference = np.nan
                        if null_status == 0:
                            if null_tangent[0] != 1.0 or not np.isfinite(null_tangent).all():
                                raise ArithmeticError("La tangente C viola kt=1 o finitud")
                            null_terms = np.array([
                                null_metric[0,0], 2*null_metric[0,3]*null_tangent[3],
                                null_metric[1,1]*null_tangent[1]**2,
                                null_metric[2,2]*null_tangent[2]**2,
                                null_metric[3,3]*null_tangent[3]**2])
                            null_residual_relative = abs(float(np.sum(null_terms)))/float(np.sum(abs(null_terms)))
                            if null_first_tangent is None:
                                null_first_tangent = null_tangent.copy()
                            null_scale_difference = float(np.max(abs(null_tangent-null_first_tangent)
                                / np.maximum(1.0, abs(null_first_tangent))))
                        null_grid_rows.append({
                            "M": null_mass, "a/M": null_spin_ratio,
                            "radio": null_radius_label, "r/M": null_radius/null_mass,
                            "theta": null_theta, "dirección": null_direction_label,
                            "escala_dirección": null_direction_scale,
                            "región": "ergosfera" if null_c_numeric > 0 else "exterior_ergosfera",
                            "raíces_positivas": null_positive_root_count,
                            "estado_C": null_status, "rho": null_residual_relative,
                            "error_invariancia_escala": null_scale_difference})
null_grid = pd.DataFrame(null_grid_rows)
null_grid_summary = (null_grid.groupby(["M", "a/M", "radio", "región"], sort=True)
    .agg(casos=("estado_C", "size"),
         válidos=("estado_C", lambda values: int((values == 0).sum())),
         sin_raíz=("estado_C", lambda values: int((values == 1).sum())),
         dos_raíces_rechazadas=("estado_C", lambda values: int((values == 2).sum())),
         max_residual_relativo=("rho", "max"),
         max_error_invariancia_escala=("error_invariancia_escala", "max"))
    .reset_index())
null_max_residual = float(null_grid["rho"].max())
null_max_scale_error = float(null_grid["error_invariancia_escala"].max())
if not np.isfinite(null_max_residual) or null_max_residual > 1e-12:
    raise ArithmeticError(f"Residual nulo fuera de tolerancia: {null_max_residual:.17g}")
if null_max_scale_error > 1e-12:
    raise ArithmeticError(f"La escala cambia la semirrecta: {null_max_scale_error:.17g}")
null_grid_summary


,M,a/M,radio,región,casos,válidos,sin_raíz,dos_raíces_rechazadas,max_residual_relativo,max_error_invariancia_escala
0,1.0,0.00,cerca_horizonte,exterior_ergosfera,210,210,0,0,2.169821e-16,0.0
1,1.0,0.00,intermedio,exterior_ergosfera,210,210,0,0,1.665335e-16,0.0
2,1.0,0.00,lejano,exterior_ergosfera,210,210,0,0,1.734723e-16,0.0
3,1.0,0.50,cerca_horizonte,ergosfera,90,0,72,18,NaN,NaN
4,1.0,0.50,cerca_horizonte,exterior_ergosfera,120,120,0,0,2.683961e-16,0.0
5,1.0,0.50,intermedio,exterior_ergosfera,210,210,0,0,1.902280e-16,0.0
6,1.0,0.50,lejano,exterior_ergosfera,210,210,0,0,3.541713e-16,0.0
7,1.0,0.90,cerca_horizonte,ergosfera,90,0,72,18,NaN,NaN
8,1.0,0.90,cerca_horizonte,exterior_ergosfera,120,120,0,0,2.441556e-16,0.0
9,1.0,0.90,intermedio,exterior_ergosfera,210,210,0,0,2.438530e-16,0.0


In [27]:
# Los rechazos geométricos permanecen visibles y no se cuentan como tangentes válidas.
null_rejected_summary = (null_grid.loc[null_grid["estado_C"] != 0]
    .groupby(["a/M", "región", "dirección"], sort=True)
    .agg(casos_rechazados=("estado_C", "size"),
         sin_raíz=("estado_C", lambda values: int((values == 1).sum())),
         dos_raíces=("estado_C", lambda values: int((values == 2).sum())))
    .reset_index())
null_rejected_summary


,a/M,región,dirección,casos_rechazados,sin_raíz,dos_raíces
0,0.50,ergosfera,azimutal_contrarrotante,18,18,0
1,0.50,ergosfera,azimutal_corrotante,18,0,18
2,0.50,ergosfera,casi_azimutal_contrarrotante,18,18,0
3,0.50,ergosfera,casi_azimutal_corrotante,18,0,18
4,0.50,ergosfera,mixta_contrarrotante,18,18,0
5,0.50,ergosfera,mixta_corrotante,18,18,0
6,0.50,ergosfera,polar_negativa,18,18,0
7,0.50,ergosfera,polar_positiva,18,18,0
8,0.50,ergosfera,radial_entrante,18,18,0
9,0.50,ergosfera,radial_saliente,18,18,0


In [28]:
# Ejemplos de la ambigüedad: ambas raíces dan k^t=1 y norma nula.
null_ambiguity_rows = []
for null_spin_ratio in null_spins[1:]:
    null_mass = 1.0
    null_spin = null_spin_ratio
    null_horizon = 1.0 + np.sqrt(1.0-null_spin**2)
    null_radius = (null_horizon+2.0)/2.0  # Horizonte < r < límite estático ecuatorial.
    null_coordinates = np.array([0.0, null_radius, np.pi/2, 0.0])
    null_metric = native_metric(null_mass, null_spin, null_coordinates)
    null_a_numeric = null_metric[3,3]
    null_b_numeric = 2*null_metric[0,3]
    null_c_numeric = null_metric[0,0]
    null_d_numeric = null_b_numeric**2 - 4*null_a_numeric*null_c_numeric
    null_q_numeric = -0.5*(null_b_numeric+np.copysign(np.sqrt(null_d_numeric), null_b_numeric))
    null_roots = sorted([null_q_numeric/null_a_numeric, null_c_numeric/null_q_numeric])
    null_inverse_tt = null_metric[3,3]/(null_metric[0,0]*null_metric[3,3]-null_metric[0,3]**2)
    for null_branch, null_root in enumerate(null_roots, start=1):
        null_terms = np.array([null_c_numeric, null_b_numeric*null_root,
                               null_a_numeric*null_root**2])
        null_ambiguity_rows.append({"a/M": null_spin_ratio, "r/M": null_radius,
                                   "g_tt": null_c_numeric, "g^tt": null_inverse_tt,
                                   "rama": null_branch, "s": null_root, "k^t": 1.0,
                                   "E=-k_t": -null_c_numeric-null_metric[0,3]*null_root,
                                   "futura": bool(null_inverse_tt < 0 and null_root > 0),
                                   "candidato": "rechazada (§6.13)",
                                   "rho": abs(np.sum(null_terms))/np.sum(abs(null_terms))})
null_ambiguity_table = pd.DataFrame(null_ambiguity_rows)
for null_spin_ratio in null_ambiguity_table["a/M"].unique():
    null_status_check, _ = candidate_null_tangent(
        1.0, null_spin_ratio,
        np.array([0.0, ((1.0+np.sqrt(1.0-null_spin_ratio**2))+2.0)/2.0, np.pi/2, 0.0]),
        np.array([0.0, 0.0, 1.0]))
    if null_status_check != 2:
        raise ArithmeticError("El candidato no rechaza una dirección con dos raíces")
if not null_ambiguity_table["futura"].all() or null_ambiguity_table["rho"].max() > 1e-12:
    raise ArithmeticError("Los ejemplos no demuestran dos raíces futuras nulas")
null_ambiguity_table


,a/M,r/M,g_tt,g^tt,rama,s,k^t,E=-k_t,futura,candidato,rho
0,0.50,1.933013,0.034654,-35.226176,1,0.040087,1.0,-0.013916,True,rechazada (§6.13),1.045612e-17
1,0.50,1.933013,0.034654,-35.226176,2,0.203636,1.0,0.070692,True,rechazada (§6.13),0.000000e+00
2,0.90,1.717945,0.164182,-14.455047,1,0.101457,1.0,-0.057879,True,rechazada (§6.13),0.000000e+00
3,0.90,1.717945,0.164182,-14.455047,2,0.343990,1.0,0.196239,True,rechazada (§6.13),7.700887e-17
4,0.99,1.570534,0.273452,-15.362086,1,0.150784,1.0,-0.083356,True,rechazada (§6.13),3.650201e-17
5,0.99,1.570534,0.273452,-15.362086,2,0.386287,1.0,0.213547,True,rechazada (§6.13),0.000000e+00
6,1.00,1.500000,0.333333,-18.333333,1,0.181818,1.0,-0.090909,True,rechazada (§6.13),0.000000e+00
7,1.00,1.500000,0.333333,-18.333333,2,0.400000,1.0,0.200000,True,rechazada (§6.13),1.040834e-16


El máximo de la malla se muestra en `null_grid_summary`; las direcciones rechazadas y los pares de raíces se muestran por separado. Esta es una prueba de construcción inicial en un dominio finito, no una validación de trayectorias ni una garantía global. La invariancia frente a la magnitud de la dirección también se verifica, pero no define un marco local de observador.

**Pendiente para el coordinador:** decidir el alcance de la exploración de ramas del candidato experimental. El contrato público de §6.13 rechaza direcciones con dos raíces futuras y el candidato lo implementa con el código `2`. La elección de rama para una futura ampliación y el marco local de observador siguen pendientes. El experimento no modifica producción, no cierra la decisión 7 de §14 y conserva separadas estas pruebas iniciales de la comparación y el microbenchmark de §§5.8–5.9.


## 5.11. Criterios antes de adoptar

Antes de integrar el candidato se requiere:

1. Resolver explícitamente la decisión 7 de §14 y conservar un generador reproducible, versiones y referencia simbólica auditables.
2. Ampliar la comparación de 5.8 frente al **RHS corregido** de producción, cubriendo masa explícita, Schwarzschild, Kerr extremo, polos y horizonte con las guardas productivas, además de tangentes temporales y nulas.
3. Repetir y ampliar 5.9 con compilador, flags, hardware, distribución de entradas, dispersión y tolerancias. El conteo CSE no prueba ganancia de rendimiento.
4. Validar trayectorias completas, deriva de norma e invariantes, eventos y errores, no solo evaluaciones puntuales.
5. Mantener el rechazo de dos raíces futuras en la ergorregión (§6.13, cerrado) y confirmar el tratamiento de la raíz doble $D=0$, aquí aceptada como raíz única (suposición experimental). La elección de rama y el marco local de observador siguen **pendientes**.
6. Definir el transporte y validación de buffers, propagación de errores y compatibilidad con la API existente antes de cualquier cambio de runtime.

La recomendación siguiente se calcula con los números de esta ejecución; no cambia el estado de ninguna decisión.


In [29]:
from IPython.display import Markdown, display
benchmark_index = benchmark_table.set_index("path")
validated_production_ratio = float(benchmark_index.loc["candidate_validated", "median_ns"]
                                  / benchmark_index.loc["production_public", "median_ns"])
def timing_text(path):
    row = benchmark_index.loc[path]
    return f"{row.median_ns:.6f} ± {row.iqr_ns:.6f} ns/evaluación"
recommendation_text = (
    "**Recomendación (no contrato).** Conservar el candidato como experimento auditable y "
    "**no adoptarlo en producción**. En los 540 intentos de §5.8, el error relativo máximo "
    f"por componente es **{rhs_max_component_relative:.17e}** y el normado "
    f"**{rhs_max_norm_relative:.17e}**; por tipo, temporal: "
    f"{rhs_by_type.loc['timelike', 'max_component_relative']:.17e} / "
    f"{rhs_by_type.loc['timelike', 'max_norm_relative']:.17e}, nulo: "
    f"{rhs_by_type.loc['null', 'max_component_relative']:.17e} / "
    f"{rhs_by_type.loc['null', 'max_norm_relative']:.17e} (componente / norma). "
    f"La meta 1e-12 queda **{rhs_numeric_status}**; el diagnóstico a 80 dígitos de §5.8 "
    "expone las cancelaciones cerca del horizonte extremo. En §5.9, mediana ± IQR "
    "(± indica dispersión, no incertidumbre ni intervalo de confianza): producción pública "
    f"**{timing_text('production_public')}**, candidato crudo "
    f"**{timing_text('candidate_raw')}**, candidato validado "
    f"**{timing_text('candidate_validated')}**. Razón mediana validado/producción "
    f"**{validated_production_ratio:.6f}**. Estas cifras describen una medición y una malla "
    "finitas; no justifican retirar la referencia, ni garantizan trayectorias ni permiten "
    "adopción automática. La decisión 7 de §14 sigue **PENDIENTE**. La ambigüedad de "
    "dos raíces futuras en la ergosfera de §5.10 se rechaza en el candidato, como exige §6.13; "
    "la elección de rama y el marco local de observador siguen pendientes. "
    "Revisión humana **pending**. Impacto documental: solo este cuaderno; "
    "sin cambios de arquitectura, API, ABI ni backend."
)
display(Markdown(recommendation_text))


**Recomendación (no contrato).** Conservar el candidato como experimento auditable y **no adoptarlo en producción**. En los 540 intentos de §5.8, el error relativo máximo por componente es **1.83931617264098929e-09** y el normado **5.45341004891153403e-10**; por tipo, temporal: 6.54909116906166122e-10 / 5.45341004891153403e-10, nulo: 1.83931617264098929e-09 / 2.21977260147731343e-10 (componente / norma). La meta 1e-12 queda **FAIL**; el diagnóstico a 80 dígitos de §5.8 expone las cancelaciones cerca del horizonte extremo. En §5.9, mediana ± IQR (± indica dispersión, no incertidumbre ni intervalo de confianza): producción pública **116.173798 ± 0.516357 ns/evaluación**, candidato crudo **144.435181 ± 0.727448 ns/evaluación**, candidato validado **167.329926 ± 0.464569 ns/evaluación**. Razón mediana validado/producción **1.440341**. Estas cifras describen una medición y una malla finitas; no justifican retirar la referencia, ni garantizan trayectorias ni permiten adopción automática. La decisión 7 de §14 sigue **PENDIENTE**. La ambigüedad de dos raíces futuras en la ergosfera de §5.10 se rechaza en el candidato, como exige §6.13; la elección de rama y el marco local de observador siguen pendientes. Revisión humana **pending**. Impacto documental: solo este cuaderno; sin cambios de arquitectura, API, ABI ni backend.

In [30]:
production_hashes_after = production_hashes()
if production_hashes_after != production_hashes_before:
    raise RuntimeError("Cambió una fuente C productiva leída/compilada durante el experimento")
status_after = subprocess.check_output(['git', '-C', str(ROOT/'relatipy'), 'status', '--short'], text=True)
validation_table = pd.DataFrame([
    {'validación': 'métrica simbólica == transcripción C, 16 componentes', 'resultado': 'PASS', 'valor': str(metric_residuals)},
    {'validación': 'métrica nativa vs SymPy', 'resultado': 'PASS', 'valor': f'{metric_native_table.max_error_absoluto.max():.17e}'},
    {'validación': 'Christoffel: fórmula/simetría, 64 componentes', 'resultado': 'PASS', 'valor': 0},
    {'validación': 'contracción simétrica y expansión CSE', 'resultado': 'PASS', 'valor': str(symbolic_residuals)},
    {'validación': 'aceleración C99 estricto desde archivo temporal', 'resultado': 'PASS', 'valor': ' '.join(strict_acceleration_command)},
    {'validación': 'aceleración C99 vs lambdify', 'resultado': 'PASS', 'valor': f'{native_acceleration_table.error_escalado.max():.17e}'},
    {'validación': f'tangente nula C99: {len(null_grid)} casos, {int(null_grid.estado_C.eq(0).sum())} válidos', 'resultado': 'PASS', 'valor': f'{null_max_residual:.17e}'},
    {'validación': '5.8 ejecución completa, status/finitud', 'resultado': 'PASS', 'valor': f'{len(comparison_cases)} intentos; {int(comparison_cases.production_status.notna().sum())} evaluados; {int(comparison_cases.production_status.isna().sum())} omisiones'},
    {'validación': '5.8 meta <=1e-12 por componente y norma', 'resultado': rhs_numeric_status, 'valor': f'{rhs_max_component_relative:.17e} / {rhs_max_norm_relative:.17e}; diagnóstico 80 dígitos visible'},
    {'validación': '5.9 benchmark C, tres caminos', 'resultado': 'PASS', 'valor': '; '.join(path+': '+timing_text(path) for path in benchmark_index.index)},
    {'validación': 'SHA256 idénticos, seis fuentes C productivas', 'resultado': 'PASS', 'valor': json.dumps(production_hashes_after, sort_keys=True)},
    {'validación': 'git -C relatipy status --short (informativo)', 'resultado': 'INFO', 'valor': f'antes={status_before!r}; después={status_after!r}'},
    {'validación': 'adopción / revisión humana', 'resultado': 'PENDIENTE', 'valor': 'no adoptado; manual_review=pending'},
])
validation_table


,validación,resultado,valor
0,"métrica simbólica == transcripción C, 16 compo...",PASS,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,métrica nativa vs SymPy,PASS,1.77635683940025046e-15
2,"Christoffel: fórmula/simetría, 64 componentes",PASS,0
3,contracción simétrica y expansión CSE,PASS,"(0, 0, 0, 0)"
4,aceleración C99 estricto desde archivo temporal,PASS,/usr/bin/cc -std=c99 -pedantic -Wall -Wextra -...
5,aceleración C99 vs lambdify,PASS,4.43239542035096228e-16
6,"tangente nula C99: 6300 casos, 5580 válidos",PASS,3.54171292346706191e-16
7,"5.8 ejecución completa, status/finitud",PASS,540 intentos; 524 evaluados; 16 omisiones
8,5.8 meta <=1e-12 por componente y norma,FAIL,1.83931617264098929e-09 / 5.45341004891153403e...
9,"5.9 benchmark C, tres caminos",PASS,production_public: 116.173798 ± 0.516357 ns/ev...


In [31]:
pd.DataFrame([{"source_id": "S-RHS", "fuente": path, "SHA256_inicio": digest,
    "SHA256_final": production_hashes_after[path], "idéntico": digest == production_hashes_after[path]}
    for path, digest in production_hashes_before.items()])


,source_id,fuente,SHA256_inicio,SHA256_final,idéntico
0,S-RHS,relatipy/native/src/geodesic/kerr.c,4621edf40b67ff97a028343fa05baf4d2fdaf239d60293...,4621edf40b67ff97a028343fa05baf4d2fdaf239d60293...,True
1,S-RHS,relatipy/native/src/geodesic/physic/kerr.c,6c19e59c8e108bac1e80cbce7bcc1d393872fa3911424f...,6c19e59c8e108bac1e80cbce7bcc1d393872fa3911424f...,True
2,S-RHS,relatipy/native/src/metric/kerr.c,7774f763b0299a570879d3b4c95ff634bc7cb32f320cff...,7774f763b0299a570879d3b4c95ff634bc7cb32f320cff...,True
3,S-RHS,relatipy/native/src/metric/physic/kerr.c,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...,4fddf9f8ea9cb624672dbd206af46a1c7d5a48347439b0...,True
4,S-RHS,relatipy/native/src/utils/numeric.c,1be3f5ded6a2a6c6241dd9c566501069377b0a30b065c9...,1be3f5ded6a2a6c6241dd9c566501069377b0a30b065c9...,True
5,S-RHS,relatipy/native/src/utils/tensor.c,d7343e23a047a2a5f602999fed95ab2728412022b9b1dc...,d7343e23a047a2a5f602999fed95ab2728412022b9b1dc...,True


In [32]:
pd.DataFrame([{"momento": "inicio", "git status --short (informativo)": status_before},
              {"momento": "final", "git status --short (informativo)": status_after}])


,momento,git status --short (informativo)
0,inicio,M bindings/cython/_core.pyx\n M native/src/ge...
1,final,M bindings/cython/_core.pyx\n M native/src/ge...
